# บทที่ 5 การถดถอยและแบบจำลองที่มีข้อจำกัดทางฟิสิกส์: โน้ตบุ๊กสำหรับชั้นเรียน
โน้ตบุ๊กประกอบตำรา *วิทยาศาสตร์ข้อมูลและการเรียนรู้ของเครื่องสำหรับนักฟิสิกส์*

โน้ตบุ๊กนี้ใช้ในชั้นเรียนได้ด้วยไฟล์นี้ไฟล์เดียว ไม่ต้องดึงโค้ดจาก GitHub และไม่ต้องดาวน์โหลดข้อมูล เนื้อหาเรียงตามบทที่ 5 ในตำรา แบ่งเป็นสี่คาบ คาบละประมาณ 60 นาที ตามแผนเวลาในคู่มือผู้สอน หัวข้อที่มีป้าย **โค้ดที่ 5.N** ตรงกับตัวอย่างโค้ดเลขเดียวกันในตำรา

## วัตถุประสงค์
เมื่อทำโน้ตบุ๊กนี้จนจบ ผู้เรียนควรสามารถ

1. ประมาณพารามิเตอร์ของกฎทางฟิสิกส์ด้วยวิธีกำลังสองน้อยสุดสามัญ (ordinary least squares, OLS) และรายงานพร้อมความคลาดเคลื่อนมาตรฐาน (standard error) และช่วงความเชื่อมั่น
2. อธิบายสมมติฐานของทฤษฎีบทเกาส์–มาร์คอฟ (Gauss–Markov theorem) และความหมายที่ถูกต้องของ $R^2$
3. ใช้ลำดับขั้นการเรียนรู้ของเครื่องหกขั้นและไปป์ไลน์ (pipeline) ของ scikit-learn โดยไม่ให้ข้อมูลรั่วไหล
4. เลือกความซับซ้อนของแบบจำลองด้วยการแลกเปลี่ยนระหว่างอคติกับความแปรปรวน (bias–variance trade-off) การปรับลงโทษ (regularization) และการตรวจสอบไขว้ (cross-validation)
5. ปรับเส้นโค้งไม่เชิงเส้นด้วยไคกำลังสองโดยใส่ขอบเขตทางฟิสิกส์ และประเมินแบบจำลองด้วยส่วนเหลือ (residual) และไคกำลังสองลดทอน (reduced chi-squared)
6. แพร่ความไม่แน่นอนไปยังปริมาณที่คำนวณต่อด้วยเมทริกซ์ความแปรปรวนร่วม (covariance matrix) และบอกได้ว่าเมื่อใดสหสัมพันธ์ทำให้ความไม่แน่นอนเพิ่มหรือลด

## ข้อมูลในโน้ตบุ๊กนี้เป็นข้อมูลจำลองทั้งหมด
ข้อมูลทุกชุดสร้างขึ้นภายในโค้ดของแต่ละหัวข้อ ด้วยเมล็ดสุ่ม 2026 ชุดเดียวกับตำรา ตัวเลขที่ได้จึงตรงกับที่ยกมาในบท

- **สปริงตามกฎของฮุก:** $k = 5.0$ N/m, 50 จุดในช่วง 0–0.20 m, สัญญาณรบกวน 0.03 N
- **สปริงไม่เป็นฮาร์มอนิก:** $k_1 = 5$ N/m, $k_3 = 500$ N/m$^3$, 60 จุดในช่วง $\pm 0.15$ m, สัญญาณรบกวน 0.05 N
- **ออสซิลเลเตอร์แบบหน่วง:** $A = 1.0$ m, $\gamma = 0.3$ s$^{-1}$, $\omega = 6.0$ rad/s, $\phi = 0.5$ rad, 200 จุดใน 10 s, สัญญาณรบกวน 0.05 m
- **ไดโอด:** $I_0 = 2$ pA, $n = 1.4$ (และ $R_s = 20~\Omega$ ในแบบฝึกหัด E5.6), สัญญาณรบกวนร้อยละ 5 ของกระแส

หัวกราฟที่มาจากข้อมูลจำลองมีคำว่า SYNTHETIC กำกับ ตัวเลขใช้เพื่อการเรียนวิธีการ ไม่ใช่ผลการวิจัย

## วิธีใช้
1. รันเซลล์ในส่วนที่ 0 ก่อนเสมอ ถ้าเรียนต่อจากคาบก่อนหลังจากปิด Colab ไปแล้ว ให้คลิกเซลล์หัวคาบที่จะเรียน แล้วเลือก **Runtime → Run before** เพื่อรันทุกเซลล์ก่อนหน้าใหม่ ใช้เวลาไม่ถึงหนึ่งนาที
2. อ่านคำอธิบาย แล้วรันเซลล์โค้ดถัดไปทีละเซลล์ด้วย `Shift + Enter` หรือใช้ Runtime → Run all ต้องรันตามลำดับจากบนลงล่าง เพราะหลายหัวข้อใช้ผลจากหัวข้อก่อนหน้า
3. ใช้หน่วยประมวลผลกลาง (CPU) ตามค่าเริ่มต้น ไม่ต้องเปลี่ยนเป็น GPU
4. เซลล์ที่มีเครื่องหมาย ✏️ เป็นแบบฝึกที่ต้องแก้โค้ดหรือเติมคำตอบเอง ส่วนเซลล์ ✅ ใช้ตรวจคำตอบ เซลล์แบบฝึกที่ยังไม่ได้เติมจะแสดงข้อความเตือน แต่ไม่ทำให้ Run all หยุด
5. ถ้าเปิดโน้ตบุ๊กจากลิงก์ GitHub บน Colab จะมีคำเตือนว่าโน้ตบุ๊กไม่ได้สร้างโดย Google ให้กด **Run anyway**

> 🔒 แนวคำตอบถูกล็อกไว้จนกว่าผู้เรียนจะตอบข้อนั้นครบ ผู้สอนเปิดแนวคำตอบทั้งหมดได้ด้วยตัวเลือก `SHOW_ANSWERS` ในส่วนที่ 0

> ป้ายกำกับในกราฟเขียนเป็นภาษาอังกฤษ เพราะแบบอักษรเริ่มต้นของ matplotlib บน Colab แสดงอักษรไทยไม่ได้

## สารบัญ

| คาบ | ส่วน | เนื้อหา | หัวข้อในตำรา | เวลาโดยประมาณ |
|---|---|---|---|---|
| — | 0 | เตรียมไลบรารี | — | 3 นาที |
| 1 | 1 | กำลังสองน้อยสุดสามัญและสมการปกติ | 5.1.1–5.1.2 | 15 นาที |
| 1 | 2 | สกัดค่าคงที่สปริงด้วย scikit-learn (โค้ดที่ 5.1) และการตีความ | 5.1.3–5.1.4 | 15 นาที |
| 1 | 3 | ความคลาดเคลื่อนมาตรฐาน ช่วงความเชื่อมั่น และ $R^2$ (โค้ดที่ 5.2) | 5.1.5 | 20 นาที |
| 1 | 4 | ความหมายของจุดตัดแกน (E5.1) | 5.1.4 | 10 นาที |
| 2 | 5 | ลำดับขั้นหกขั้นและการปรับมาตราส่วน (E5.2) | 5.2.1–5.2.2 | 15 นาที |
| 2 | 6 | ลักษณะเด่นจากฟิสิกส์ ภาวะร่วมเส้นตรงพหุ และกับดักของลอการิทึม | 5.2.3 | 12 นาที |
| 2 | 7 | ไปป์ไลน์และการรั่วไหลของข้อมูล (โค้ดที่ 5.3) | 5.2.4 | 8 นาที |
| 2 | 8 | การถดถอยพหุนามและอคติกับความแปรปรวน (โค้ดที่ 5.4) | 5.3.1–5.3.2 | 15 นาที |
| 2 | 9 | เส้นโค้งดีกรี 1, 3, 6 และ 10 (E5.3) | 5.3.2 | 10 นาที |
| 3 | 10 | การปรับลงโทษแบบริดจ์และแลสโซ | 5.3.3 | 10 นาที |
| 3 | 11 | การตรวจสอบไขว้ (โค้ดที่ 5.5) และเกณฑ์สารสนเทศ | 5.3.4–5.3.5 | 15 นาที |
| 3 | 12 | การปรับเส้นโค้งด้วยไคกำลังสอง: ออสซิลเลเตอร์แบบหน่วง (โค้ดที่ 5.6) | 5.4.1–5.4.2 | 15 นาที |
| 3 | 13 | ค่าเริ่มต้นจากกราฟและตัวประกอบคุณภาพ (E5.5) | 5.4.2, 5.4.5 | 20 นาที |
| 4 | 14 | ขอบเขตทางฟิสิกส์และสมการของช็อกลีย์ (โค้ดที่ 5.7) | 5.4.3–5.4.5 | 20 นาที |
| 4 | 15 | การวิเคราะห์ส่วนเหลือ | 5.5.1 | 12 นาที |
| 4 | 16 | ไคกำลังสองลดทอน | 5.5.2 | 13 นาที |
| 4 | 17 | การแพร่ของความคลาดเคลื่อนด้วยเมทริกซ์ความแปรปรวนร่วม (โค้ดที่ 5.8) | 5.5.3 | 15 นาที |
| นอกชั้น | 18 | E5.2 และ E5.5 ส่วนที่เหลือ, E5.4, E5.6 และ E5.7 | 5.2–5.5 | — |

---
# ส่วนที่ 0 เตรียมไลบรารี

เซลล์ถัดไปสร้างโฟลเดอร์ทำงาน `ch05_regression` แล้วย้ายเข้าไปทำงานในโฟลเดอร์นั้น ถ้าเปิดโน้ตบุ๊กจากโฟลเดอร์ `code/notebooks/` ในเครื่องของตนเอง เซลล์จะใช้โฟลเดอร์ `code/ch05_regression/` ที่มีอยู่แล้ว

บทนี้เริ่มใช้ไลบรารี scikit-learn สำหรับการเรียนรู้ของเครื่อง และใช้ statsmodels ต่อจากบทที่ 4 สำหรับการอนุมานของสัมประสิทธิ์ Colab มีทั้งสองตัวอยู่แล้ว เซลล์จะติดตั้งให้เฉพาะตัวที่ยังไม่มี ข้อมูลของบทนี้สร้างขึ้นในโค้ดของแต่ละหัวข้อ จึงไม่มีขั้นสร้างข้อมูลแยก

In [ ]:
import os, sys, subprocess, importlib.util

if os.path.basename(os.getcwd()) != "ch05_regression":
    if os.path.isdir("../ch05_regression"):        # เปิดจาก code/notebooks/ ในเครื่องของตนเอง
        os.chdir("../ch05_regression")
    else:                                           # Colab หรือโฟลเดอร์ว่าง
        os.makedirs("ch05_regression", exist_ok=True)
        os.chdir("ch05_regression")
for d in ("data", "outputs"):
    os.makedirs(d, exist_ok=True)
print("โฟลเดอร์ทำงาน:", os.getcwd())

_need = {"sklearn": "scikit-learn", "statsmodels": "statsmodels"}
missing = [pip for mod, pip in _need.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
print("ติดตั้งเพิ่ม:", missing if missing else "ไม่มี (มีครบแล้ว)")

### การล็อกแนวคำตอบ
เซลล์ ✅ และเซลล์แนวคำตอบในโน้ตบุ๊กนี้แสดงคำตอบอ้างอิงเมื่อผู้เรียนเติมคำตอบในเซลล์ ✏️ ของข้อนั้นครบแล้วเท่านั้น ถ้ายังไม่ได้ตอบ เซลล์จะแสดงเพียงข้อความ 🔒 บอกว่ายังขาดคำตอบใด การคำนวณในเซลล์ยังทำงานตามปกติ ดังนั้น Run all จึงรันได้ครบโดยไม่เปิดเผยคำตอบ แนวคำตอบของงานเขียนและงานนอกชั้นซึ่งตรวจด้วยโค้ดไม่ได้จะเปิดเมื่อผู้สอนเลือกเท่านั้น

**สำหรับผู้สอน:** ติ๊กช่อง `SHOW_ANSWERS` ในเซลล์ถัดไป (หรือแก้เป็น `True`) แล้วรันเซลล์นั้นอีกครั้ง เพื่อแสดงแนวคำตอบทุกข้อ เช่น เมื่อเฉลยในชั้นหรือเตรียมสอน

In [ ]:
#@title ⚙️ ตัวเลือกของผู้สอน: แสดงแนวคำตอบ
SHOW_ANSWERS = False  #@param {type:"boolean"}

def _answered(v):
    """True when an exercise variable holds an answer (not None, "?", "", empty or containing blanks)."""
    if v is None or v is Ellipsis:
        return False
    if isinstance(v, str):                       # placeholders such as "?", "?-?", "..." count as blank
        return v.replace("?", "").replace("-", "").replace(".", "").strip() != ""
    if isinstance(v, dict):
        return len(v) > 0 and all(_answered(x) for x in v.values())
    if isinstance(v, (list, tuple, set, frozenset)):
        return len(v) > 0 and all(_answered(x) for x in v)
    if type(v).__name__ == "ndarray":
        return v.size > 0
    return True

def _gate(names=()):
    """Return print if the answers may be shown, otherwise print one lock message and return a no-op."""
    if SHOW_ANSWERS:
        return print
    if names:
        missing = [n for n in names if not _answered(globals().get(n))]
        if not missing:
            return print
        print("🔒 ยังไม่ได้ตอบ:", ", ".join(missing),
              "— เติมคำตอบในเซลล์ ✏️ ด้านบนให้ครบก่อน แล้วรันเซลล์นี้อีกครั้ง")
    else:
        print("🔒 แนวคำตอบข้อนี้ผู้สอนเป็นผู้เปิด (ตั้ง SHOW_ANSWERS = True ในส่วนที่ 0)")
    return lambda *args, **kwargs: None

print("แสดงแนวคำตอบทั้งหมด:", "ใช่ (โหมดผู้สอน)" if SHOW_ANSWERS else "ไม่ (แสดงเมื่อผู้เรียนตอบข้อนั้นครบแล้ว)")

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy import stats
import sklearn
import statsmodels

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 140)

for name, mod in [("Python", sys), ("numpy", np), ("scipy", scipy), ("pandas", pd),
                  ("scikit-learn", sklearn), ("statsmodels", statsmodels)]:
    ver = sys.version.split()[0] if name == "Python" else mod.__version__
    print(f"{name:12s} {ver}")

---
# คาบที่ 1 กลศาสตร์และการถดถอยเชิงเส้น

> **เริ่มคาบนี้หลังจากปิด Colab ไปแล้ว:** คลิกเซลล์นี้ แล้วเลือก Runtime → Run before เพื่อรันส่วนที่ 0 ใหม่ก่อนเรียนต่อ

---
# ส่วนที่ 1 กำลังสองน้อยสุดสามัญและสมการปกติ (หัวข้อ 5.1.1–5.1.2)

### แนวคิดสำคัญ
กฎของฮุก (Hooke's law) บอกว่าแรงที่ต้องใช้ดึงสปริงให้ยืดออกระยะ $x$ คือ $F = kx$ บทนี้วัดแรงที่ผู้ทดลองกระทำ จึงใช้เครื่องหมายบวก ส่วน $F = -kx$ ในตำราหลายเล่มคือแรงที่สปริงดึงกลับ ถ้าไม่ระบุให้ชัดว่า $F$ คือแรงใด จะตีความเครื่องหมายของ $k$ ที่ได้ผิด

ถ้าวัดคู่ $(x_i, F_i)$ มา $N$ คู่และการวัดแรงมีสัญญาณรบกวน เราต้องการเกณฑ์ที่ทุกคนใช้ร่วมกันได้ในการวาง "เส้นที่ผ่านกลางข้อมูลได้ดีที่สุด" วิธีกำลังสองน้อยสุดสามัญ (ordinary least squares, OLS) ใช้เกณฑ์ว่า ผลรวมกำลังสองของส่วนเหลือ (residual) $r_i = y_i - \hat y_i$ ต้องน้อยที่สุด

$$J(\boldsymbol\theta) = \|\mathbf{y} - \mathbf{X}\boldsymbol\theta\|^2
\quad\Longrightarrow\quad
\hat{\boldsymbol\theta} = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$$

เมื่อ $\mathbf{X}$ คือเมทริกซ์ออกแบบ (design matrix) ซึ่งแต่ละแถวเป็นการวัดหนึ่งครั้ง และมีหลักที่เป็นเลขหนึ่งสำหรับจุดตัดแกน สมการทางขวาเรียกว่าสมการปกติ (normal equation)

### ทำไมต้องเป็นกำลังสอง
ถ้าสัญญาณรบกวนแจกแจงแบบเกาส์และมีขนาดเท่ากันทุกจุด ความน่าจะเป็นของข้อมูลชุดที่เห็นแปรตาม $\exp[-\sum_i r_i^2/2\sigma^2]$ การทำให้ $\sum r_i^2$ น้อยที่สุดจึงเท่ากับการประมาณภาวะน่าจะเป็นสูงสุด (maximum likelihood estimation)

เซลล์ถัดไปใช้ข้อมูลเล็ก ๆ ห้าจุด ลองหมุนเส้นตรงหลายความชันแล้วดูว่า $J$ เปลี่ยนอย่างไร จากนั้นเทียบกับผลเฉลยของสมการปกติ

In [ ]:
toy_x = np.array([0.00, 0.05, 0.10, 0.15, 0.20])          # m
toy_F = np.array([0.02, 0.24, 0.49, 0.77, 0.99])          # N (made-up numbers)
toy_X = np.column_stack([np.ones_like(toy_x), toy_x])      # design matrix: [1, x]

toy_theta = np.linalg.solve(toy_X.T @ toy_X, toy_X.T @ toy_F)   # normal equation
toy_r = toy_F - toy_X @ toy_theta
print(f"normal equation: intercept = {toy_theta[0]:.4f} N, slope = {toy_theta[1]:.4f} N/m")
print(f"X^T r = {toy_X.T @ toy_r}  (residuals are orthogonal to every column of X)")
print(f"sum of residuals = {toy_r.sum():.1e}  (zero because the model has an intercept)")

slopes = np.linspace(4.0, 6.0, 201)
J = [np.sum((toy_F - (toy_F.mean() - s * toy_x.mean()) - s * toy_x)**2) for s in slopes]
plt.figure(figsize=(6, 3.2))
plt.plot(slopes, J); plt.axvline(toy_theta[1], ls=":", color="r", label="normal equation")
plt.xlabel("slope k (N/m)"); plt.ylabel(r"J = sum of squared residuals (N$^2$)")
plt.title("least squares: J has one minimum (toy data)"); plt.legend(); plt.tight_layout(); plt.show()

สำหรับแบบจำลองที่เป็นเชิงเส้นในพารามิเตอร์ $J$ เป็นพาราโบลาของพารามิเตอร์ จึงมีจุดต่ำสุดจุดเดียวและหาได้ด้วยสูตรปิด ส่วนเหลือที่ได้ตั้งฉากกับทุกหลักของ $\mathbf{X}$ แปลว่าข้อมูลเข้าไม่มีส่วนใดที่ยังอธิบายส่วนเหลือได้อีก

### สมมติฐานของ OLS
| สมมติฐาน | ข้อความทางสถิติ | ความหมายในการทดลอง |
|---|---|---|
| *เงื่อนไขของทฤษฎีบทเกาส์–มาร์คอฟ* | | |
| ความเป็นเชิงเส้น | $E[y\mid x]$ เป็นเชิงเส้นในพารามิเตอร์ | กฎทางฟิสิกส์ใช้ได้ในช่วงที่วัด |
| ค่าเฉลี่ยศูนย์ | $E[\epsilon_i] = 0$ | ไม่มีความคลาดเคลื่อนเชิงระบบที่ยังไม่ได้ใส่ในแบบจำลอง |
| ความไม่สัมพันธ์กัน | $\mathrm{Cov}(\epsilon_i,\epsilon_j) = 0$ | ไม่มีการเลื่อนของเครื่องมือตามเวลา |
| ความแปรปรวนคงที่ | $\mathrm{Var}(\epsilon_i) = \sigma^2$ | สัญญาณรบกวนไม่ขึ้นกับขนาดของค่าที่วัด |
| *เงื่อนไขเพิ่มเติมสำหรับการอนุมานแบบแม่นตรง* | | |
| ความเป็นเกาส์ | $\epsilon_i \sim \mathcal{N}(0,\sigma^2)$ | สัญญาณรบกวนเกิดจากแหล่งเล็ก ๆ จำนวนมาก |

ทฤษฎีบทเกาส์–มาร์คอฟบอกว่า ภายใต้สี่ข้อแรก OLS เป็นตัวประมาณเชิงเส้นที่ไม่เอนเอียงและมีความแปรปรวนต่ำที่สุด ทฤษฎีบทนี้ **ไม่ได้** ต้องการความเป็นเกาส์ ความเป็นเกาส์จำเป็นเมื่อต้องการให้ค่าพีและช่วงความเชื่อมั่นแม่นตรงเมื่อ $N$ น้อย

> **ความเข้าใจคลาดเคลื่อนที่พบบ่อย:** "ข้อมูลต้องแจกแจงแบบเกาส์จึงจะใช้การถดถอยเชิงเส้นได้" สิ่งที่ต้องพิจารณาคือการแจกแจงของ *สัญญาณรบกวน* ไม่ใช่ของ $x$ หรือ $y$ ในการทดลองสปริง ผู้ทดลองเลือก $x$ ห่างเท่า ๆ กัน $x$ จึงแจกแจงแบบสม่ำเสมอ แต่การถดถอยยังใช้ได้ดีทุกประการ

---
# ส่วนที่ 2 สกัดค่าคงที่สปริงด้วย scikit-learn (หัวข้อ 5.1.3–5.1.4)

scikit-learn ออกแบบให้แบบจำลองทุกชนิดใช้ด้วยรูปแบบเดียวกัน คือสร้างแบบจำลอง เรียก `fit(X, y)` แล้วเรียก `predict(X_new)` โค้ดที่ 5.1 สร้างข้อมูลสปริงที่ $k = 5.0$ N/m วัด 50 จุดตั้งแต่ 0 ถึง 0.20 m ด้วยสัญญาณรบกวน 0.03 N ข้อควรสังเกตคือ scikit-learn ต้องการข้อมูลเข้าเป็นอาร์เรย์สองมิติ แม้จะมีตัวแปรเดียว จึงต้องใช้ `reshape(-1, 1)`

### โค้ดที่ 5.1 — การถดถอยเชิงเส้นของข้อมูลสปริงตามกฎของฮุก

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(seed=2026)
k_true = 5.0                                   # N/m
x = np.linspace(0.0, 0.20, 50)                 # m
F_obs = k_true * x + rng.normal(scale=0.03, size=x.size)   # N

X = x.reshape(-1, 1)                           # shape (50, 1)
model = LinearRegression()
model.fit(X, F_obs)
print(f"k_fit = {model.coef_[0]:.4f} N/m, intercept = {model.intercept_:.4f} N")
print(f"R^2 = {model.score(X, F_obs):.4f}")

In [ ]:
xx = np.linspace(0, 0.20, 100).reshape(-1, 1)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(x, F_obs, "o", ms=4, label="data")
ax[0].plot(xx, model.predict(xx), "r", label=f"OLS: k = {model.coef_[0]:.3f} N/m")
ax[0].plot(xx, k_true * xx, "k:", label="true law k = 5.0 N/m")
ax[0].set_xlabel("stretch x (m)"); ax[0].set_ylabel("force F (N)"); ax[0].legend(fontsize=8)
ax[0].set_title("Hooke's law (SYNTHETIC)")
ax[1].plot(x, F_obs - model.predict(X), "o", ms=4); ax[1].axhline(0, color="k", lw=0.8)
ax[1].set_xlabel("stretch x (m)"); ax[1].set_ylabel("residual (N)"); ax[1].set_title("residuals")
plt.tight_layout(); plt.show()

ผลคือ $k_{\text{fit}} \approx 5.174$ N/m จุดตัดแกนราว $-0.011$ N และ $R^2 \approx 0.991$ ส่วนเหลือกระจายแบบสุ่มรอบศูนย์ แต่ตัวเลขชุดนี้ยังไม่พอ ผลที่ไม่มีความไม่แน่นอนกำกับเป็นผลที่ไม่สมบูรณ์ เราไม่รู้ว่า 5.174 ห่างจาก 5.0 มากหรือน้อยเมื่อเทียบกับความสามารถของข้อมูลชุดนี้ ส่วนที่ 3 จะเติมส่วนที่ขาด

### ✏️ แบบฝึก 5.A สมการปกติด้วยมือ
สร้างเมทริกซ์ออกแบบ `my_Xd` ของข้อมูลสปริง (หลักแรกเป็นเลขหนึ่ง หลักที่สองเป็น `x`) แล้วแก้สมการปกติด้วย `np.linalg.solve` เก็บผลใน `my_theta` ซึ่งควรเป็นอาร์เรย์ `[จุดตัดแกน, ความชัน]`

In [ ]:
my_Xd = None        # ✏️ np.column_stack([...])
my_theta = None     # ✏️ np.linalg.solve(..., ...)

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_theta'])
if my_theta is None:
    _say("(ยังไม่ได้ตอบ)")
else:
    ok = np.allclose(my_theta, [model.intercept_, model.coef_[0]], rtol=1e-8, atol=1e-10)
    _say("✔ ตรงกับ scikit-learn" if ok else "✘ ยังไม่ตรง", "—",
          f"scikit-learn: intercept {model.intercept_:.4f} N, slope {model.coef_[0]:.4f} N/m")
    _say("scikit-learn ใช้วิธีเชิงตัวเลขที่เสถียรกว่าการหาเมทริกซ์ผกผันตรง ๆ แต่ได้คำตอบเดียวกัน")

### การตีความค่าสัมประสิทธิ์ในเชิงฟิสิกส์
ความชันคือค่า $k$ โดยตรง ผู้ทดลองที่ดีจะตรวจไขว้ด้วยวิธีอื่น เช่น วัดคาบการแกว่งของมวลที่แขวนกับสปริง ซึ่งให้ $k = 4\pi^2 m/T^2$ ถ้าสองวิธีต่างกันเกินความไม่แน่นอน แปลว่ามีความคลาดเคลื่อนเชิงระบบซ่อนอยู่

ตามกฎของฮุก แรงที่ระยะยืดศูนย์ต้องเป็นศูนย์ จุดตัดแกนที่ไม่เป็นศูนย์จึงต้องถามว่าเกิดจากสัญญาณรบกวนหรือจากฟิสิกส์

| ลักษณะของจุดตัดแกน | คำอธิบายทางฟิสิกส์ที่เป็นไปได้ |
|---|---|
| เล็กเมื่อเทียบกับความไม่แน่นอนของตัวเอง | สัญญาณรบกวนแบบสุ่ม ไม่มีนัยสำคัญ |
| ใหญ่และเป็นบวก | สปริงมีแรงตึงเริ่มต้น หรือตั้งจุดอ้างอิงของระยะยืดผิด |
| ใหญ่และเป็นลบ | เซนเซอร์วัดแรงมีค่าชดเชย (offset) ที่ไม่ได้ปรับศูนย์ หรือจุดอ้างอิงผิดไปอีกทาง |
| เปลี่ยนไปเมื่อเปลี่ยนช่วงของการวัด | แบบจำลองเชิงเส้นไม่พอ เช่น สปริงเริ่มไม่เป็นฮาร์มอนิก |

### ✏️ แบบฝึก 5.B การถดถอยไม่รู้จักหน่วย
สมมติว่าผู้ทดลองบันทึกระยะยืดเป็นเซนติเมตร (`x_cm = 100 * x`) แล้วทำการถดถอยด้วยข้อมูลแรงชุดเดิม จงคำนวณความชันที่ได้ เก็บใน `my_slope_cm` (หน่วย N/cm) แล้วแปลงกลับเป็น N/m เก็บใน `my_k_from_cm`

In [ ]:
x_cm = 100 * x
my_slope_cm = None      # ✏️ ใช้ LinearRegression().fit(x_cm.reshape(-1, 1), F_obs).coef_[0]
my_k_from_cm = None     # ✏️ แปลงหน่วย N/cm -> N/m

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_k_from_cm', 'my_slope_cm'])
_s = model.coef_[0] / 100
_say("ความชัน (N/cm):", "(ยังไม่ได้ตอบ)" if my_slope_cm is None else
      ("✔" if abs(my_slope_cm / _s - 1) < 1e-6 else f"✘ ค่าที่คาดคือ {_s:.5f} N/cm"))
_say("แปลงกลับ (N/m):", "(ยังไม่ได้ตอบ)" if my_k_from_cm is None else
      ("✔" if abs(my_k_from_cm / model.coef_[0] - 1) < 1e-6 else f"✘ ค่าที่คาดคือ {model.coef_[0]:.4f} N/m"))
_say("ถ้ารายงาน 0.0517 ว่าเป็น N/m ค่าจะผิดไปหนึ่งร้อยเท่า การตรวจหน่วยเป็นหน้าที่ของผู้วิเคราะห์ทั้งหมด")

---
# ส่วนที่ 3 ความคลาดเคลื่อนมาตรฐาน ช่วงความเชื่อมั่น และ R² (หัวข้อ 5.1.5)

ถ้าทำการทดลองซ้ำกับสปริงตัวเดิม จะได้ข้อมูลชุดใหม่และค่า $k$ ใหม่ ความคลาดเคลื่อนมาตรฐานบอกว่า $k$ จะกระจายเพียงใดถ้าทำซ้ำหลายครั้ง ภายใต้สมมติฐานของ OLS

$$\mathrm{Cov}(\hat{\boldsymbol\theta}) = \sigma^2(\mathbf{X}^\top\mathbf{X})^{-1},\qquad
\hat\sigma^2 = \frac{1}{N-p}\sum_i (y_i-\hat y_i)^2,\qquad
\mathrm{SE}(\hat\theta_1) = \frac{\sigma}{\sqrt{\sum_i (x_i-\bar x)^2}}$$

ตัวหาร $N-p$ มีเหตุผลเดียวกับตัวหาร $N-1$ ในบทที่ 4 สูตรทางขวาบอกเรื่องการออกแบบการทดลอง การวัดที่ $x$ กระจายกว้างให้ความชันแม่นกว่า เหมือนช่างไม้ที่วัดความเอียงจากปลายสองข้างของไม้กระดาน

scikit-learn เน้นการทำนาย จึงไม่คำนวณความคลาดเคลื่อนมาตรฐานให้ ไลบรารี statsmodels ให้ผลการอนุมานครบ

### โค้ดที่ 5.2 — การอนุมานของสัมประสิทธิ์ด้วย statsmodels

In [ ]:
import statsmodels.api as sm

X_ic = sm.add_constant(X)             # add the column of ones for the intercept
ols_model = sm.OLS(F_obs, X_ic).fit()
print(ols_model.summary())            # coef, SE, t, p, 95% CI
print(ols_model.bse)                  # standard errors only
print(ols_model.conf_int(alpha=0.05)) # 95% confidence intervals

In [ ]:
k_hat, se_k = ols_model.params[1], ols_model.bse[1]
lo, hi = ols_model.conf_int(alpha=0.05)[1]
print(f"k = {k_hat:.3f} +/- {se_k:.3f} N/m, 95 % CI [{lo:.2f}, {hi:.2f}] N/m  ->  report k = (5.17 +/- 0.07) N/m")
print(f"k_true = 5.0 lies {(k_hat - k_true) / se_k:.1f} SE from the estimate; inside the 95 % CI? {lo <= k_true <= hi}")
print(f"check the SE formula: sigma_hat / sqrt(sum (x - xbar)^2) = "
      f"{np.sqrt(ols_model.scale) / np.sqrt(np.sum((x - x.mean())**2)):.4f} N/m")

ค่าจริง 5.0 N/m อยู่ **นอก** ช่วงความเชื่อมั่นร้อยละ 95 ห่างจากค่าที่ได้ราว $2.5$ เท่าของความคลาดเคลื่อนมาตรฐาน นี่ไม่ใช่ข้อผิดพลาดของโค้ด แต่เป็นตัวอย่างจริงของอุปมาการโยนห่วงในบทที่ 4 วิธีสร้างช่วงความเชื่อมั่นร้อยละ 95 จะพลาดราวหนึ่งครั้งในยี่สิบครั้ง และข้อมูลชุดนี้บังเอิญเป็นหนึ่งในครั้งเหล่านั้น (จึงไม่ควรเปลี่ยนเมล็ดสุ่มเพื่อ "แก้" ผลนี้)

เซลล์ถัดไปทำการทดลองซ้ำ 2,000 ครั้งด้วยเมล็ดสุ่มอื่น แล้วนับว่าช่วงความเชื่อมั่นพลาดค่าจริงกี่ครั้ง

In [ ]:
miss, ci_list = 0, []
for s in range(2000):
    r = np.random.default_rng(s)
    F = k_true * x + r.normal(scale=0.03, size=x.size)
    lo_s, hi_s = sm.OLS(F, X_ic).fit().conf_int(alpha=0.05)[1]
    miss += not lo_s <= k_true <= hi_s
    ci_list.append((lo_s, hi_s))
print(f"2,000 repeated experiments: the 95 % CI misses k_true in {miss / 2000:.3f} of them")

ci = np.array(ci_list[:40])
bad = (ci[:, 0] > k_true) | (ci[:, 1] < k_true)
plt.figure(figsize=(7, 3.2))
for i, (a, b) in enumerate(ci):
    plt.plot([i, i], [a, b], color="r" if bad[i] else "C0", lw=2)
plt.axhline(k_true, color="k", ls=":"); plt.xlabel("repeated experiment"); plt.ylabel("95 % CI of k (N/m)")
plt.title("ring toss: red intervals miss k_true (SYNTHETIC)"); plt.tight_layout(); plt.show()

ในการทดลองจริงเราไม่รู้ค่าจริง จึงไม่มีทางรู้ว่าผลของเราเป็นหนึ่งในครั้งที่พลาดหรือไม่ นี่คือเหตุผลที่การยืนยันผลด้วยการทดลองซ้ำหรือด้วยวิธีอื่นมีความสำคัญ

### R² สูงไม่ได้แปลว่าแบบจำลองถูกหรือค่าที่ได้แม่นยำ
ข้อมูลสปริงข้างต้นให้ $R^2 = 0.991$ แต่ $k$ คลาดจากค่าจริงราวร้อยละ 3.5 และค่าจริงอยู่นอกช่วงความเชื่อมั่น เซลล์ถัดไป (ตัวอย่างเพิ่มเติมจากตำรา) สร้างสปริงที่มีพจน์ไม่เป็นฮาร์มอนิกเล็กน้อย $F = 5x + 10x^3$ ด้วยสัญญาณรบกวนเท่าเดิม แล้วปรับด้วยเส้นตรง

In [ ]:
rng_r2 = np.random.default_rng(2026)
F_soft = 5.0 * x + 10.0 * x**3 + rng_r2.normal(scale=0.03, size=x.size)   # slightly anharmonic
fit_soft = sm.OLS(F_soft, X_ic).fit()
print(f"straight line on F = 5x + 10x^3: R^2 = {fit_soft.rsquared:.4f}, "
      f"slope = {fit_soft.params[1]:.3f} +/- {fit_soft.bse[1]:.3f} N/m")
print(f"the initial stiffness k1 = 5.0 N/m lies {(fit_soft.params[1] - 5) / fit_soft.bse[1]:.1f} SE away")

res = fit_soft.resid
smooth = np.convolve(res, np.ones(7) / 7, mode="same")
plt.figure(figsize=(7, 3))
plt.plot(x, res, "o", ms=4, alpha=0.6, label="residual")
plt.plot(x[3:-3], smooth[3:-3], "r", label="running mean of 7 points")
plt.axhline(0, color="k", lw=0.8); plt.xlabel("stretch x (m)"); plt.ylabel("residual (N)")
plt.title("R^2 = 0.99 but the residuals curve (SYNTHETIC)"); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

$R^2$ ยังสูงถึง 0.99 แต่ความชันที่ได้คือความแข็งเฉลี่ยในช่วงที่วัด ไม่ใช่ $k_1$ และห่างจาก 5.0 หลายเท่าของความคลาดเคลื่อนมาตรฐาน ร่องรอยของปัญหาอยู่ในส่วนเหลือ ซึ่งเป็นบวกที่ปลายทั้งสองข้างและเป็นลบตรงกลาง

สรุปคือ $R^2$ บอกเพียงสัดส่วนของความแปรปรวนของ $y$ ที่เส้นอธิบายได้ และขึ้นกับช่วงของ $x$ ที่วัดมาก ความแม่นยำของพารามิเตอร์ต้องดูจากความคลาดเคลื่อนมาตรฐาน ส่วนความถูกต้องของรูปแบบแบบจำลองต้องดูจากส่วนเหลือ (ส่วนที่ 15)

### ✏️ แบบฝึก 5.C ช่วงของการวัดกับความแม่นยำ (คำถามทบทวนที่ 2)
นิสิตสองคนวัดสปริงตัวเดียวกันด้วย 50 จุดและสัญญาณรบกวน 0.03 N เท่ากัน คนแรกวัดในช่วง 0–5 cm คนที่สองวัดในช่วง 0–20 cm ใช้สูตร $\mathrm{SE} = \sigma/\sqrt{\sum(x_i-\bar x)^2}$ คำนวณความคลาดเคลื่อนมาตรฐานของทั้งสองคน แล้วเก็บอัตราส่วน $\mathrm{SE}_{5\,\text{cm}}/\mathrm{SE}_{20\,\text{cm}}$ ใน `my_ratio`

In [ ]:
x5, x20 = np.linspace(0, 0.05, 50), np.linspace(0, 0.20, 50)
se5 = None          # ✏️
se20 = None         # ✏️
my_ratio = None     # ✏️ se5 / se20

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_ratio'])
_se = {w: 0.03 / np.sqrt(np.sum((v - v.mean())**2)) for w, v in ((5, x5), (20, x20))}
if my_ratio is None:
    _say("(ยังไม่ได้ตอบ)")
else:
    _say("✔" if abs(my_ratio - _se[5] / _se[20]) < 0.01 else "✘", "—",
          f"SE: 0-5 cm {_se[5]:.3f} N/m, 0-20 cm {_se[20]:.3f} N/m, ratio {_se[5] / _se[20]:.1f}")
_say("ช่วงกว้างขึ้นสี่เท่า SE เล็กลงราวสี่เท่า แต่ช่วงที่กว้างขึ้นต้องยังอยู่ในบริเวณที่กฎของฮุกใช้ได้")

---
# ส่วนที่ 4 ความหมายของจุดตัดแกน (หัวข้อ 5.1.4)

### ✏️ แบบฝึก E5.1 ความหมายทางฟิสิกส์ของจุดตัดแกนที่ไม่เป็นศูนย์ (5 นาที)
นิสิตคนหนึ่งทดลองกับสปริงอีกตัวหนึ่ง และได้ผลการถดถอย $k = (4.95 \pm 0.05)$ N/m และจุดตัดแกน $(0.20 \pm 0.03)$ N

1. คำนวณว่าจุดตัดแกนห่างจากศูนย์กี่เท่าของความคลาดเคลื่อนมาตรฐาน เก็บใน `my_z` และตอบว่าต่างจากศูนย์อย่างมีนัยสำคัญหรือไม่ใน `my_significant` (`True` หรือ `False`)
2. เขียนคำอธิบายทางฟิสิกส์อย่างน้อยสองข้อ และการทดลองที่แยกคำอธิบายเหล่านั้นออกจากกันได้ ในเซลล์ข้อความถัดไป

In [ ]:
my_z = None             # ✏️
my_significant = None   # ✏️ True หรือ False

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_significant', 'my_z'])
_say("z:", "(ยังไม่ได้ตอบ)" if my_z is None else ("✔" if abs(my_z - 0.20 / 0.03) < 0.05 else "✘ ค่าที่คาดคือ 0.20/0.03 = 6.7"))
_say("มีนัยสำคัญ:", "(ยังไม่ได้ตอบ)" if my_significant is None else
      ("✔" if my_significant is True else "✘ ห่างจากศูนย์ 6.7 SE จึงมีนัยสำคัญอย่างชัดเจน"))

**คำอธิบายทางฟิสิกส์และการทดลองที่แยกกัน (E5.1):**

- คำอธิบายที่ 1: ...
- คำอธิบายที่ 2: ...
- การทดลองที่แยกสองคำอธิบาย: ...
- ค่า $k$ ยังใช้ได้หรือไม่ ถ้าสาเหตุเป็นแต่ละข้อ: ...

In [ ]:
#@markdown แนวคำตอบ E5.1 (รันหลังตอบเองแล้ว)
_say = _gate(['my_significant', 'my_z'])
_say('''แนวคำตอบ E5.1
- จุดตัดแกนห่างจากศูนย์ 0.20/0.03 = 6.7 SE จึงมีนัยสำคัญอย่างชัดเจน
- คำอธิบายที่เป็นบวก: (1) สปริงมีแรงตึงเริ่มต้น เช่น สปริงขดแน่น  (2) เซนเซอร์วัดแรงมีค่าชดเชย
- การทดลองแยก: อ่านค่าเซนเซอร์โดยไม่ติดสปริง ถ้าอ่านได้ราว 0.2 N ปัญหาอยู่ที่เซนเซอร์
  ถ้าอ่านได้ศูนย์ ให้เปลี่ยนเป็นสปริงที่รู้ว่าไม่มีแรงตึงเริ่มต้น ถ้าจุดตัดแกนหายไป ปัญหาอยู่ที่สปริงตัวเดิม
- ถ้าสาเหตุคือค่าชดเชยของเซนเซอร์ ค่า k ยังใช้ได้ เพราะค่าชดเชยคงที่เลื่อนเส้นทั้งเส้นโดยไม่เปลี่ยนความชัน''')

---
# คาบที่ 2 ลำดับขั้นหกขั้น ไปป์ไลน์ และอคติกับความแปรปรวน

> **เริ่มคาบนี้หลังจากปิด Colab ไปแล้ว:** คลิกเซลล์นี้ แล้วเลือก Runtime → Run before เพื่อสร้างข้อมูลและตัวแปรจากคาบก่อนหน้าใหม่ก่อนเรียนต่อ

---
# ส่วนที่ 5 ลำดับขั้นหกขั้นและการปรับมาตราส่วน (หัวข้อ 5.2.1–5.2.2)

### ลำดับขั้นการเรียนรู้ของเครื่องหกขั้น
ตำราใช้ลำดับขั้นนี้เป็นโครงสร้างร่วมของทุกวิธี ตั้งแต่การถดถอยในบทนี้จนถึงโครงข่ายประสาทเทียม

| ขั้น | งาน | ตัวอย่างสปริง |
|---|---|---|
| 1 กำหนดปัญหา | คำถามทางฟิสิกส์ ปริมาณเป้าหมาย เกณฑ์ว่าคำตอบดีพอ | วัด $k$ ให้แม่นกว่าร้อยละ 2 |
| 2 ข้อมูล | โหลด สำรวจ (บทที่ 4) แบ่งข้อมูลฝึกกับข้อมูลตรวจสอบ | 50 คู่ $(x, F)$ |
| 3 ลักษณะเด่น | สร้างจากความรู้ทางฟิสิกส์ (บทที่ 3) และปรับมาตราส่วน | $x$ หรือ $x, x^3$ |
| 4 แบบจำลอง | เลือกและปรับแบบจำลอง | OLS |
| 5 ประเมิน | MSE, $R^2$, ส่วนเหลือ, $\chi^2_{\text{red}}$, การตรวจสอบไขว้ | ส่วนเหลือสุ่มหรือไม่ |
| 6 ตีความ | ความหมายทางฟิสิกส์ของพารามิเตอร์พร้อมความไม่แน่นอน | $k = (5.17 \pm 0.07)$ N/m |

ขั้นที่ถูกข้ามบ่อยที่สุดคือขั้นที่ 1 และขั้นที่ 6 ผู้เรียนจำนวนมากเริ่มจากเลือกแบบจำลองทันที และจบที่ตัวเลขความแม่นยำ โดยไม่เคยถามว่าตัวเลขนั้นตอบคำถามทางฟิสิกส์อะไร

### การปรับให้เป็นบรรทัดฐานและการปรับให้เป็นมาตรฐาน
| ชื่อ | สูตร | เหมาะกับ |
|---|---|---|
| การปรับให้เป็นบรรทัดฐาน (normalization) | $(x-x_{\min})/(x_{\max}-x_{\min})$ ได้ค่าในช่วง $[0,1]$ | ข้อมูลที่มีขอบเขตชัดเจน เช่น ความเข้มของพิกเซล |
| การปรับให้เป็นมาตรฐาน (standardization) | $(x-\bar x)/s$ ได้ค่าเฉลี่ย 0 และส่วนเบี่ยงเบนมาตรฐาน 1 | ข้อมูลที่ไม่มีขอบเขตชัดเจน การปรับลงโทษ และการวิเคราะห์องค์ประกอบหลัก |

นักฟิสิกส์คุ้นกับความคิดนี้ในรูปของตัวแปรไร้มิติ เช่น เลขเรย์โนลด์ ต่างกันที่การทำให้ไร้มิติใช้สเกลทางฟิสิกส์ ส่วนการปรับให้เป็นมาตรฐานใช้สเกลทางสถิติของข้อมูลชุดนั้น ข้อที่ต้องระวังคือ สัมประสิทธิ์ที่ได้จากข้อมูลที่ปรับแล้วไม่ใช่ค่า $k$ ในหน่วย N/m อีกต่อไป

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
demo = pd.DataFrame({"x (m)": x[:5], "normalized": MinMaxScaler().fit_transform(X)[:5, 0],
                     "standardized": StandardScaler().fit_transform(X)[:5, 0]})
print(demo.round(4).to_string(index=False))
print(f"x: mean {x.mean():.3f} m, std (ddof=0) {x.std():.4f} m")

### ✏️ แบบฝึก E5.2 (ส่วนในชั้น 5 นาที) การถดถอยของข้อมูลดิบกับข้อมูลที่ปรับเป็นมาตรฐาน
ใช้ข้อมูลสปริงจากโค้ดที่ 5.1

1. ปรับ `X` ให้เป็นมาตรฐานด้วย `StandardScaler().fit_transform(X)` เก็บใน `Z`
2. ทำการถดถอยเชิงเส้นของ `F_obs` กับ `Z` แล้วเก็บความชันใน `my_k_std`
3. เปรียบเทียบกับ `my_k_raw * s_x` เมื่อ `s_x = x.std()` (หารด้วย $N$ แบบเดียวกับ `StandardScaler`)

ส่วนที่สองของแบบฝึกนี้ (ทำไมการปรับลงโทษต้องปรับมาตราส่วนก่อน) อยู่ในส่วนที่ 18

In [ ]:
my_k_raw = model.coef_[0]      # from code 5.1
Z = None                       # ✏️
my_k_std = None                # ✏️

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_k_raw', 'my_k_std'])
_kstd = my_k_raw * x.std()
if my_k_std is None:
    _say("(ยังไม่ได้ตอบ)")
else:
    _say("✔" if abs(my_k_std / _kstd - 1) < 1e-6 else "✘", "—",
          f"k_std = {_kstd:.4f} N (unit of F, because z has no unit); k_raw * s_x (ddof=0) = {_kstd:.4f} N")
_say(f"with ddof=1: k_raw * s_x = {my_k_raw * x.std(ddof=1):.4f} N -> differs slightly, StandardScaler divides by N")
_say("reason: substitute x = xbar + s_x z into F = c + k x  ->  slope with respect to z is k s_x")

---
# ส่วนที่ 6 ลักษณะเด่นจากฟิสิกส์ ภาวะร่วมเส้นตรงพหุ และกับดักของลอการิทึม (หัวข้อ 5.2.3)

การถดถอยเชิงเส้นต้องการให้แบบจำลองเป็นเชิงเส้นใน **พารามิเตอร์** ไม่จำเป็นต้องเป็นเชิงเส้นใน **ตัวแปร** ความรู้ทางฟิสิกส์จึงแปลงหลายปัญหาให้เป็นปัญหาเชิงเส้นได้

| ระบบ | ตัวแปรเดิม | ลักษณะเด่นใหม่ | เหตุผล |
|---|---|---|---|
| การตกอิสระไม่มีแรงต้าน | $t$ | $t,\ t^2$ | $y = y_0 + v_0t - \tfrac12 gt^2$ เป็นเชิงเส้นใน $y_0, v_0, g$ |
| สปริงไม่เป็นฮาร์มอนิก | $x$ | $x,\ x^3$ | $F = k_1x + k_3x^3$ |
| สมการอาร์เรเนียส | $T$ | $1/T$ (เป้าหมาย $\ln k_{\text{rate}}$) | $\ln k_{\text{rate}} = \ln A - E_a/(RT)$ |
| กฎกำลัง $y = ax^n$ | $x$ | $\ln x$ (เป้าหมาย $\ln y$) | $\ln y = \ln a + n\ln x$ |
| การสลายแบบเอกซ์โพเนนเชียล | $t$ | $t$ (เป้าหมาย $\ln y$) | $\ln y = \ln y_0 - t/\tau$ |
| การแกว่งที่รู้ความถี่แล้ว | $t$ | $\cos\omega t,\ \sin\omega t$ | เป็นเชิงเส้นใน $a, b$ เมื่อรู้ $\omega$ |

แถวสุดท้ายมีเงื่อนไข ถ้ายังไม่รู้ $\omega$ หรืออัตราการหน่วง แบบจำลองจะไม่เป็นเชิงเส้นในพารามิเตอร์ และต้องใช้การปรับเส้นโค้งไม่เชิงเส้นในคาบที่ 3

### ✏️ แบบฝึก 5.D เชิงเส้นในพารามิเตอร์หรือไม่ (คำถามทบทวนที่ 1)
เติม `True` ถ้าแบบจำลองปรับได้ด้วย OLS (อาจต้องสร้างลักษณะเด่นก่อน) และ `False` ถ้าต้องใช้การปรับเส้นโค้งไม่เชิงเส้น พารามิเตอร์ที่ต้องหาคือตัวอักษร $a, b, c$

In [ ]:
my_linear = {
    "ก: y = a + b exp(-t/5)": None,     # ✏️
    "ข: y = a + b exp(-t/c)": None,     # ✏️
    "ค: y = a t + b t^2": None,         # ✏️
    "ง: y = a cos(c t) + b sin(c t)": None,   # ✏️
}

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_linear'])
_lin = {"ก: y = a + b exp(-t/5)": (True, "ลักษณะเด่น exp(-t/5) คำนวณได้จาก t จึงเป็นเชิงเส้นใน a, b"),
        "ข: y = a + b exp(-t/c)": (False, "c อยู่ในเลขชี้กำลัง"),
        "ค: y = a t + b t^2": (True, "ลักษณะเด่น t และ t^2"),
        "ง: y = a cos(c t) + b sin(c t)": (False, "ความถี่ c ไม่รู้ จึงไม่เป็นเชิงเส้นใน c")}
for k_, (v, why) in _lin.items():
    mine = my_linear[k_]
    _say(f"{k_:32s}", "(ยังไม่ได้ตอบ)" if mine is None else ("✔" if mine == v else "✘"), "—", why)

### ภาวะร่วมเส้นตรงพหุ
ลักษณะเด่นที่มากเกินไปก็เป็นปัญหาได้ ถ้าลักษณะเด่นสองตัวสัมพันธ์กันสูง เช่น $x$ กับ $x^2$ เมื่อ $x$ เป็นบวกทั้งหมด เมทริกซ์ $\mathbf{X}^\top\mathbf{X}$ จะใกล้เอกฐาน และความคลาดเคลื่อนมาตรฐานของสัมประสิทธิ์จะพองขึ้นมาก ปัญหานี้เรียกว่าภาวะร่วมเส้นตรงพหุ (multicollinearity) ตัวชี้วัดคือตัวประกอบการพองของความแปรปรวน (variance inflation factor, VIF) ซึ่งถ้าเกินราว 10 ถือว่ารุนแรง

In [ ]:
def vif(*cols):
    M = np.column_stack(cols)
    M = (M - M.mean(0)) / M.std(0)
    return np.diag(np.linalg.inv(np.corrcoef(M.T)))

z = (x - x.mean()) / x.std()
print(f"x = 0-0.20 m: corr(x, x^2) = {np.corrcoef(x, x**2)[0, 1]:.3f}, VIF = {vif(x, x**2)[0]:.0f}")
print(f"x, x^2, x^3 together: VIF = {np.round(vif(x, x**2, x**3)).astype(int)}")
print(f"standardized first: corr(z, z^2) = {np.corrcoef(z, z**2)[0, 1]:.3f}, "
      f"corr(z, z^3) = {np.corrcoef(z, z**3)[0, 1]:.3f}, VIF(z, z^2, z^3) = {np.round(vif(z, z**2, z**3), 1)}")

การปรับให้เป็นมาตรฐานก่อนสร้างพหุนามทำให้สหสัมพันธ์ระหว่าง $z$ กับ $z^2$ หายไป แต่ $z$ กับ $z^3$ ยังสัมพันธ์กันราว 0.92 เพราะทั้งคู่เป็นฟังก์ชันคี่ วิธีแก้อื่นคือตัดลักษณะเด่นที่ซ้ำซ้อน หรือใช้การปรับลงโทษแบบริดจ์ในคาบที่ 3

### กับดัก: แปลงเป็นลอการิทึมแล้วปรับเส้นตรง
การแปลงทำให้ **แบบจำลอง** เป็นเส้นตรงก็จริง แต่เปลี่ยน **สัญญาณรบกวน** ไปด้วย เซลล์ถัดไปจำลองการสลายที่ $\tau = 2.0$ s ด้วยสัญญาณรบกวนคงที่ 2 จำนวนนับ 1,000 ชุด แล้วเทียบการปรับเส้นตรงของ $\ln y$ กับการปรับเส้นโค้งเดิมโดยตรงด้วย `curve_fit` (ซึ่งจะเรียนในคาบที่ 3)

In [ ]:
from scipy.optimize import curve_fit
rng_log = np.random.default_rng(2026)
t_dec = np.linspace(0, 10, 50)
tau_true, y0, sig = 2.0, 100.0, 2.0                 # constant noise in counts
tau_log, tau_cf, n_dropped = [], [], []
for _ in range(1000):
    y = y0 * np.exp(-t_dec / tau_true) + rng_log.normal(scale=sig, size=t_dec.size)
    ok = y > 0                                      # ln y impossible for y <= 0
    n_dropped.append((~ok).sum())
    tau_log.append(-1 / np.polyfit(t_dec[ok], np.log(y[ok]), 1)[0])
    p, _ = curve_fit(lambda tt, a, b: a * np.exp(-tt / b), t_dec, y, p0=[80, 1.5])
    tau_cf.append(p[1])
print(f"points with y <= 0 that ln y must drop: {np.mean(n_dropped):.1f} per data set on average")
print(f"unweighted line on ln y: tau = {np.mean(tau_log):.2f} +/- {np.std(tau_log):.2f} s (mean +/- spread)")
print(f"curve_fit on y directly: tau = {np.mean(tau_cf):.3f} +/- {np.std(tau_cf):.3f} s   (true 2.0 s)")

การปรับเส้นตรงของ $\ln y$ ให้ $\tau$ ที่เอนเอียงและกระจายกว้างกว่าเกือบห้าเท่า เพราะจุดช่วงท้ายของการสลายซึ่ง $y$ เล็กมีสัญญาณรบกวนใหญ่มากหลังแปลง และจุดที่ $y \le 0$ ต้องถูกทิ้ง การแปลงเป็นลอการิทึมยังมีประโยชน์สำหรับการสำรวจข้อมูลและการหาค่าเริ่มต้น แต่ไม่ควรใช้เป็นการปรับขั้นสุดท้ายโดยไม่ถ่วงน้ำหนัก

---
# ส่วนที่ 7 ไปป์ไลน์และการรั่วไหลของข้อมูล (หัวข้อ 5.2.4)

ไปป์ไลน์ (pipeline) ของ scikit-learn ผูกขั้นการปรับมาตราส่วน การสร้างลักษณะเด่น และการถดถอยไว้เป็นแบบจำลองเดียว การเรียก `fit` ครั้งเดียวจะทำทุกขั้นตามลำดับ

### โค้ดที่ 5.3 — ไปป์ไลน์ที่ต่อการปรับมาตราส่วน การสร้างพหุนาม และการถดถอยเข้าด้วยกัน

In [ ]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('poly',   PolynomialFeatures(degree=2, include_bias=False)),
    ('reg',    LinearRegression()),
])
pipe.fit(X, F_obs)          # X must be 2-D, as in the previous listing
F_pred = pipe.predict(X)

In [ ]:
print(f"pipeline (scale -> degree 2 -> OLS): MSE = {np.mean((F_pred - F_obs)**2):.2e} N^2")
print("steps:", list(pipe.named_steps))
print(f"the scaler inside the pipeline learned mean = {pipe.named_steps['scaler'].mean_[0]:.3f} m "
      f"and std = {pipe.named_steps['scaler'].scale_[0]:.4f} m from the data passed to fit()")

ประโยชน์ที่สำคัญที่สุดของไปป์ไลน์คือป้องกันการรั่วไหลของข้อมูล (data leakage) ถ้าคำนวณค่าเฉลี่ยและส่วนเบี่ยงเบนมาตรฐานจากข้อมูลทั้งหมด **ก่อน** แบ่งข้อมูล ข้อมูลตรวจสอบก็ได้ "แอบ" ส่งข้อมูลของตัวเองเข้าไปในขั้นการฝึกแล้ว เหมือนผู้ออกข้อสอบที่นำคะแนนของข้อสอบจริงไปปรับเกณฑ์ก่อนสอบ เมื่อใช้ไปป์ไลน์ร่วมกับการตรวจสอบไขว้ ตัวปรับมาตราส่วนจะเรียนรู้จากข้อมูลฝึกของแต่ละรอบเท่านั้นโดยอัตโนมัติ ตำราจึงใช้ไปป์ไลน์เป็นโครงสร้างมาตรฐานตั้งแต่บทนี้เป็นต้นไป

---
# ส่วนที่ 8 การถดถอยพหุนามและอคติกับความแปรปรวน (หัวข้อ 5.3.1–5.3.2)

เมื่อดึงสปริงให้ยืดมากขึ้น พจน์แก้ไขตัวแรกของสปริงที่สมมาตรคือพจน์กำลังสาม $F = k_1x + k_3x^3$ ถ้าไม่รู้รูปนี้ล่วงหน้า ทางเลือกคือการถดถอยพหุนาม (polynomial regression) ดีกรี $d$ ซึ่งยังเป็นเชิงเส้นในสัมประสิทธิ์ ค่า $d$ เป็นไฮเพอร์พารามิเตอร์ (hyperparameter) ที่ผู้วิเคราะห์ต้องเลือก ถ้าต่ำเกินไปจะเรียนรู้น้อยเกินไป (underfitting) ถ้าสูงเกินไปจะเรียนรู้เกิน (overfitting)

### การแยกส่วนอคติและความแปรปรวน
$$\mathbb{E}\big[(y-\hat f(x))^2\big] = \underbrace{\big(\mathbb{E}[\hat f(x)]-f(x)\big)^2}_{\text{bias}^2}
+ \underbrace{\mathbb{E}\big[(\hat f(x)-\mathbb{E}[\hat f(x)])^2\big]}_{\text{variance}}
+ \underbrace{\sigma^2}_{\text{irreducible}}$$

| องค์ประกอบ | สาเหตุ | ลายเซ็น |
|---|---|---|
| อคติ$^2$ (bias) | แบบจำลองง่ายเกินไป ขาดกลไกทางฟิสิกส์ | ความคลาดเคลื่อนสูงทั้งข้อมูลฝึกและข้อมูลตรวจสอบ |
| ความแปรปรวน (variance) | แบบจำลองซับซ้อนเกินไป | ต่ำในข้อมูลฝึก แต่สูงในข้อมูลตรวจสอบ |
| ส่วนที่ลดไม่ได้ | สัญญาณรบกวนของการวัด | พื้นขั้นต่ำที่ความคลาดเคลื่อนของข้อมูลตรวจสอบลงไปต่ำกว่าไม่ได้ |

อุปมาคือช่างตัดเสื้อสองคน คนแรกตัดจากแบบมาตรฐานสามขนาด (อคติสูง) คนที่สองวัดตัวละเอียดรวมถึงรอยย่นของเสื้อที่ลูกค้าใส่มาในวันนั้น (ความแปรปรวนสูง)

โค้ดที่ 5.4 สร้างข้อมูลสปริงไม่เป็นฮาร์มอนิก $k_1 = 5$ N/m, $k_3 = 500$ N/m$^3$ 60 จุดในช่วง $\pm 0.15$ m สัญญาณรบกวน 0.05 N แบ่งร้อยละ 70 ไว้ฝึก ร้อยละ 30 ไว้ตรวจสอบ แล้วลองดีกรี 1 ถึง 15

### โค้ดที่ 5.4 — กวาดดีกรีของพหุนามเพื่อดูการแลกเปลี่ยนระหว่างอคติกับความแปรปรวน

In [ ]:
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(seed=2026)
k1, k3 = 5.0, 500.0                                  # N/m, N/m^3
x_an = rng.uniform(-0.15, 0.15, 60)
F_an = k1 * x_an + k3 * x_an**3 + rng.normal(scale=0.05, size=x_an.size)
X_an = x_an.reshape(-1, 1)

X_train, X_val, y_train, y_val = train_test_split(
    X_an, F_an, test_size=0.3, random_state=42)
mse_train, mse_val = [], []
for d in range(1, 16):
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('poly',   PolynomialFeatures(degree=d, include_bias=False)),
        ('reg',    LinearRegression()),
    ])
    pipe.fit(X_train, y_train)
    mse_train.append(np.mean((pipe.predict(X_train) - y_train)**2))
    mse_val.append(np.mean((pipe.predict(X_val) - y_val)**2))

In [ ]:
print(" d   MSE_train    MSE_val   (noise variance sigma^2 = 0.0025 N^2)")
for d, a, b in zip(range(1, 16), mse_train, mse_val):
    print(f"{d:2d}   {a:9.5f}  {b:9.5f}")
print(f"lowest validation MSE at degree {int(np.argmin(mse_val)) + 1}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(X_train, y_train, "o", ms=4, label="training"); ax[0].plot(X_val, y_val, "s", ms=4, label="validation")
ax[0].set_xlabel("x (m)"); ax[0].set_ylabel("F (N)"); ax[0].legend(); ax[0].set_title("anharmonic spring (SYNTHETIC)")
ax[1].semilogy(range(1, 16), mse_train, "o-", label="training")
ax[1].semilogy(range(1, 16), mse_val, "s-", label="validation")
ax[1].axhline(0.05**2, ls=":", color="k", label=r"$\sigma^2$")
ax[1].set_xlabel("polynomial degree"); ax[1].set_ylabel(r"MSE (N$^2$)"); ax[1].legend(); ax[1].set_title("bias-variance scan")
plt.tight_layout(); plt.show()

ผลเล่าเรื่องได้ครบทั้งสามส่วน ที่ดีกรี 1 และ 2 MSE ของทั้งสองชุดอยู่ราว 0.04–0.06 N$^2$ สูงกว่า $\sigma^2$ ราวยี่สิบเท่า นี่คืออคติ เส้นตรงและพาราโบลาจับรูปตัว S ของ $x^3$ ไม่ได้ ที่ดีกรี 3 MSE ของข้อมูลตรวจสอบลดเหลือราว 0.0031 N$^2$ ใกล้พื้นขั้นต่ำ $\sigma^2 = 0.0025$ N$^2$ หลังจากนั้น MSE ของข้อมูลฝึกลดลงต่อจนเหลือราว 0.0009 N$^2$ ที่ดีกรี 15 ซึ่ง **ต่ำกว่า** $\sigma^2$ ขณะที่ MSE ของข้อมูลตรวจสอบกลับเพิ่มเป็นราว 0.007 N$^2$ แบบจำลองดีกรีสูงกำลังจำสัญญาณรบกวน

เมื่อปรับพหุนามดีกรี 3 กับข้อมูลทั้งหมดด้วย $x$ ดิบ สัมประสิทธิ์จะมีหน่วยทางฟิสิกส์

In [ ]:
fit3 = sm.OLS(F_an, np.column_stack([np.ones_like(x_an), x_an, x_an**2, x_an**3])).fit()
th, se = fit3.params, fit3.bse
print(f"degree 3 on all data: theta1 = {th[1]:.2f} +/- {se[1]:.2f} N/m, "
      f"theta2 = {th[2]:.1f} +/- {se[2]:.1f} N/m^2, theta3 = {th[3]:.0f} +/- {se[3]:.0f} N/m^3")
print(f"theta2 is {abs(th[2]) / se[2]:.1f} SE from zero -> consistent with a symmetric spring (no x^2 term)")

> **ความเข้าใจคลาดเคลื่อนที่พบบ่อย:** "ความคลาดเคลื่อนของข้อมูลฝึกยิ่งต่ำยิ่งดี" MSE ของข้อมูลฝึกลดลงเสมอเมื่อเพิ่มความซับซ้อน ค่าที่ต่ำกว่า $\sigma^2$ ที่รู้จากการวัดจึงเป็นสัญญาณเตือนของการเรียนรู้เกิน ไม่ใช่ความสำเร็จ เกณฑ์ที่ใช้เลือกแบบจำลองต้องมาจากข้อมูลที่แบบจำลองไม่เคยเห็นเสมอ

### ✏️ แบบฝึก 5.E วินิจฉัยแบบจำลอง (คำถามทบทวนที่ 3)
แบบจำลองหนึ่งให้ MSE ของข้อมูลฝึก 0.001 ของข้อมูลตรวจสอบ 0.02 ขณะที่ความแปรปรวนของสัญญาณรบกวนที่รู้จากการวัดซ้ำคือ 0.004 เลือกคำวินิจฉัย (`"ก"`, `"ข"` หรือ `"ค"`) และวิธีแก้ที่เหมาะสม (เลือกได้หลายข้อจาก `"1"`–`"4"`)

- วินิจฉัย: (ก) อคติสูง เรียนรู้น้อยเกินไป (ข) ความแปรปรวนสูง เรียนรู้เกิน (ค) แบบจำลองพอดีแล้ว
- วิธีแก้: (1) เพิ่มดีกรี (2) ลดดีกรี (3) เพิ่มการปรับลงโทษแล้วเลือกระดับด้วยการตรวจสอบไขว้ (4) เก็บข้อมูลเพิ่ม

In [ ]:
my_diagnosis = "?"      # ✏️ "ก", "ข" หรือ "ค"
my_fixes = set()        # ✏️ เช่น {"1", "4"}

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_diagnosis', 'my_fixes'])
_say("วินิจฉัย:", "(ยังไม่ได้ตอบ)" if my_diagnosis == "?" else ("✔" if my_diagnosis == "ข" else "✘"),
      "— MSE ฝึก 0.001 ต่ำกว่า sigma^2 = 0.004 แปลว่ากำลังจำสัญญาณรบกวน และ MSE ตรวจสอบสูงกว่ามาก")
_say("วิธีแก้:", "(ยังไม่ได้ตอบ)" if not my_fixes else ("✔" if my_fixes == {"2", "3", "4"} else "✘"),
      "— ลดดีกรี เพิ่มการปรับลงโทษ และการเก็บข้อมูลเพิ่มก็ลดความแปรปรวนได้ (การเพิ่มดีกรีทำให้แย่ลง)")

---
# ส่วนที่ 9 เส้นโค้งดีกรี 1, 3, 6 และ 10 (หัวข้อ 5.3.2)

### ✏️ แบบฝึก E5.3 (5 นาที) การแลกเปลี่ยนระหว่างอคติกับความแปรปรวน
เซลล์ถัดไปปรับพหุนามดีกรีตามรายการ `degrees` กับข้อมูลฝึก วาดเทียบกับกฎจริง แล้วสุ่มข้อมูลใหม่ 200 ชุดเพื่อดูว่าแต่ละเส้นเปลี่ยนไปเพียงใด (ความแปรปรวน) ที่กลางช่วง ($x = 0$) และใกล้ขอบ ($x = 0.145$ m) รันเซลล์ แล้วตอบคำถามในเซลล์ ✏️ ถัดไป

In [ ]:
degrees = [1, 3, 6, 10]          # try other degrees if you like

def make_an(seed):
    r = np.random.default_rng(seed=seed)
    xs = r.uniform(-0.15, 0.15, 60)
    Fs = 5.0 * xs + 500.0 * xs**3 + r.normal(scale=0.05, size=xs.size)
    return train_test_split(xs.reshape(-1, 1), Fs, test_size=0.3, random_state=42)

def poly_model(d):
    return Pipeline([('scaler', StandardScaler()),
                     ('poly', PolynomialFeatures(degree=d, include_bias=False)),
                     ('reg', LinearRegression())])

xg = np.linspace(-0.15, 0.15, 400).reshape(-1, 1)
truth = 5.0 * xg[:, 0] + 500.0 * xg[:, 0]**3
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(X_train, y_train, "k.", label="training data"); ax.plot(xg, truth, "k:", lw=1, label="true law")
for d in degrees:
    ax.plot(xg, poly_model(d).fit(X_train, y_train).predict(xg), label=f"degree {d}")
ax.set_ylim(-3, 3); ax.set_xlabel("x (m)"); ax.set_ylabel("F (N)"); ax.legend(fontsize=8)
ax.set_title("polynomial fits to the training data (SYNTHETIC)"); plt.tight_layout(); plt.show()

spread = {}
for d in degrees:
    pc, pe = [], []
    for s in range(200):
        Xt, _, yt, _ = make_an(10_000 + s)
        m_ = poly_model(d).fit(Xt, yt)
        pc.append(m_.predict([[0.0]])[0]); pe.append(m_.predict([[0.145]])[0])
    spread[d] = (np.std(pc), np.std(pe))
    print(f"degree {d:2d}: spread over 200 new data sets  x = 0: {spread[d][0]:.3f} N   x = 0.145 m: {spread[d][1]:.3f} N")

In [ ]:
my_best_degree = None          # ✏️ ดีกรีที่เหมาะที่สุด
my_most_variable = None        # ✏️ ดีกรีที่เส้นเปลี่ยนมากที่สุดเมื่อสุ่มข้อมูลใหม่ (ดูที่ขอบ)

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_best_degree', 'my_most_variable'])
_say("ดีกรีที่เหมาะที่สุด:", "(ยังไม่ได้ตอบ)" if my_best_degree is None else
      ("✔" if my_best_degree == 3 else "✘ ดีกรี 3 ต่ำที่สุดที่ใส่พจน์ไม่เป็นฮาร์มอนิกตัวแรกของสปริงสมมาตรได้"))
_say("ความแปรปรวนสูงสุด:", "(ยังไม่ได้ตอบ)" if my_most_variable is None else
      ("✔" if my_most_variable == max(degrees) else f"✘ ดูตัวเลขที่ x = 0.145 m ของดีกรี {max(degrees)}"))

**อธิบายด้วยภาษาของอคติกับความแปรปรวน (E5.3):** เส้นของแต่ละดีกรีผิดอย่างไร

- ดีกรี 1: ...
- ดีกรี 3: ...
- ดีกรี 6: ...
- ดีกรี 10: ...

In [ ]:
#@markdown แนวคำตอบ E5.3 (รันหลังตอบเองแล้ว)
_say = _gate(['my_best_degree', 'my_most_variable'])
_say('''แนวคำตอบ E5.3
- ดีกรี 1 เป็นเส้นตรงที่ตัดผ่านรูปตัว S ส่วนเหลือเป็นบวกและลบเป็นช่วง ๆ อย่างมีระบบ = อคติ
- ดีกรี 3 ผ่านกลางข้อมูลได้ดีตลอดช่วง และแทบไม่เปลี่ยนเมื่อสุ่มข้อมูลใหม่
- ดีกรี 6 คล้ายดีกรี 3 ในช่วงกลาง แต่เริ่มโค้งผิดปกติที่ปลาย
- ดีกรี 10 แกว่งระหว่างจุดข้อมูล โดยเฉพาะที่ขอบซึ่งมีข้อมูลน้อย = ความแปรปรวน
  ที่ x = 0.145 m เส้นดีกรี 10 กระจายราว 0.56 N เทียบกับดีกรี 3 ราว 0.03 N
- ดีกรี 3 เป็นดีกรีต่ำที่สุดที่ใส่พจน์ไม่เป็นฮาร์มอนิกตัวแรกของสปริงสมมาตรได้''')

---
# คาบที่ 3 การปรับลงโทษ การตรวจสอบไขว้ และการปรับเส้นโค้งด้วยไคกำลังสอง

> **เริ่มคาบนี้หลังจากปิด Colab ไปแล้ว:** คลิกเซลล์นี้ แล้วเลือก Runtime → Run before เพื่อสร้างข้อมูลและตัวแปรจากคาบก่อนหน้าใหม่ก่อนเรียนต่อ

---
# ส่วนที่ 10 การปรับลงโทษแบบริดจ์และแลสโซ (หัวข้อ 5.3.3)

การลดดีกรีเป็นวิธีหนึ่งในการคุมความซับซ้อน อีกวิธีหนึ่งคือใช้แบบจำลองที่ยืดหยุ่นไว้ก่อน แต่ "ลงโทษ" สัมประสิทธิ์ที่ใหญ่เกินไป เพราะการเรียนรู้เกินมักปรากฏเป็นสัมประสิทธิ์ขนาดใหญ่ที่มีเครื่องหมายสลับกัน การปรับลงโทษ (regularization) เพิ่มพจน์ที่วัดขนาดของสัมประสิทธิ์เข้าไปในฟังก์ชันเป้าหมาย

$$J_{\text{ridge}} = \sum_i (y_i-\hat y_i)^2 + \lambda\sum_{j\ge1}\theta_j^2,\qquad
J_{\text{lasso}} = \sum_i (y_i-\hat y_i)^2 + \lambda\sum_{j\ge1}|\theta_j|$$

| สมบัติ | ริดจ์ (ridge, L2) | แลสโซ (lasso, L1) |
|---|---|---|
| ผลต่อสัมประสิทธิ์ | หดทุกตัว แต่ไม่ถึงศูนย์ | บางตัวเป็นศูนย์พอดี |
| เหมาะกับ | ตัวแปรทุกตัวมีส่วนร่วม หรือสัมพันธ์กันสูง | ต้องการคัดเลือกตัวแปร |
| ผลเฉลย | รูปปิด $(\mathbf{X}^\top\mathbf{X}+\lambda\mathbf{I})^{-1}\mathbf{X}^\top\mathbf{y}$ | ต้องหาเชิงตัวเลข |
| บริเวณจำกัดในสองมิติ | วงกลม | รูปเพชร (มุมอยู่บนแกน) |

**มุมมองของนักฟิสิกส์:** การทำให้ผลรวมกำลังสองของส่วนเหลือน้อยที่สุดภายใต้เงื่อนไข $\sum_j\theta_j^2 \le C$ แก้ได้ด้วยตัวคูณลากรานจ์ $\mathcal{L} = \sum_i(y_i-\hat y_i)^2 + \lambda(\sum_j\theta_j^2 - C)$ และ $\lambda$ ก็คือตัวคูณลากรานจ์นั่นเอง ริดจ์ยังเป็นกรณีเฉพาะของการปรับลงโทษแบบทีโคนอฟ (Tikhonov regularization) ที่ใช้แก้ปัญหาผกผันในฟิสิกส์ เช่น การกู้ภาพจากภาพเบลอ

เซลล์ถัดไป (ตัวอย่างเพิ่มเติมจากตำรา) ปรับพหุนามดีกรี 10 กับข้อมูลฝึกของส่วนที่ 8 ด้วยสามวิธี แล้วเทียบสัมประสิทธิ์ของลักษณะเด่น $z, z^2, \ldots, z^{10}$ ที่ปรับเป็นมาตรฐานแล้ว

In [ ]:
from sklearn.linear_model import Ridge, Lasso

def poly10(reg):
    return Pipeline([('scale_x', StandardScaler()),
                     ('poly', PolynomialFeatures(degree=10, include_bias=False)),
                     ('scale_f', StandardScaler()), ('reg', reg)])

rows = {}
for name, reg in (("OLS", LinearRegression()), ("ridge, alpha = 0.3", Ridge(alpha=0.3)),
                  ("lasso, alpha = 0.001", Lasso(alpha=1e-3, max_iter=200_000))):
    p_ = poly10(reg).fit(X_train, y_train)
    c_ = p_.named_steps["reg"].coef_
    rows[name] = list(np.round(c_, 2)) + [int(np.sum(np.abs(c_) > 1e-8)), np.mean((p_.predict(X_val) - y_val)**2)]
cols = [f"z^{j}" for j in range(1, 11)] + ["non-zero", "MSE_val"]
print(pd.DataFrame(rows, index=cols).T.to_string(float_format=lambda v: f"{v:.4g}"))

OLS ให้สัมประสิทธิ์ของพจน์กำลังสูงขนาดใหญ่ที่มีเครื่องหมายสลับกัน ซึ่งเป็นลายเซ็นของการเรียนรู้เกิน ริดจ์หดทุกตัวลงจนเหลือเพียง $z$ กับ $z^3$ ที่มีขนาดสำคัญ แต่ไม่มีตัวใดเป็นศูนย์พอดี (ค่า 0 ในตารางเป็นผลของการปัดเศษสองตำแหน่ง คอลัมน์ non-zero นับจากค่าก่อนปัดเศษ) ส่วนแลสโซตั้งสัมประสิทธิ์ส่วนใหญ่เป็นศูนย์ ทั้งสองวิธีให้ MSE ของข้อมูลตรวจสอบใกล้กับพหุนามดีกรี 3 โดยไม่ต้องบอกดีกรีที่ถูกล่วงหน้า

> **ข้อระวัง:** พจน์ลงโทษวัดขนาดของสัมประสิทธิ์ ซึ่งขึ้นกับหน่วยของตัวแปร ก่อนใช้ริดจ์หรือแลสโซจึงต้องปรับตัวแปรทุกตัวให้เป็นมาตรฐานก่อนเสมอ (E5.2 ส่วนที่สองในส่วนที่ 18 แสดงตัวอย่างเชิงตัวเลข)

---
# ส่วนที่ 11 การตรวจสอบไขว้และเกณฑ์สารสนเทศ (หัวข้อ 5.3.4–5.3.5)

การแบ่งข้อมูลครั้งเดียวให้ผลที่ขึ้นกับว่าจุดใดบังเอิญไปอยู่ในชุดตรวจสอบ (validation set) การตรวจสอบไขว้แบบ $k$ ส่วน (k-fold cross-validation) แบ่งข้อมูลเป็น $k$ ส่วน ในรอบที่ $i$ ใช้ส่วนที่ $i$ ตรวจสอบและส่วนที่เหลือฝึก ทำครบ $k$ รอบแล้วเฉลี่ยความคลาดเคลื่อน

โค้ดที่ 5.5 ใช้การตรวจสอบไขว้ห้าส่วนเลือก $\lambda$ ของริดจ์สำหรับพหุนามดีกรี 10 ไปป์ไลน์มีตัวปรับมาตราส่วนสองตัว ตัวแรกปรับ $x$ ก่อนสร้างพหุนามเพื่อไม่ให้ $x^{10}$ เล็กจนเกิดปัญหาเชิงตัวเลข ตัวที่สองปรับลักษณะเด่นทุกตัวหลังสร้างพหุนาม เพื่อให้การลงโทษยุติธรรมกับทุกพจน์ scikit-learn เรียก $\lambda$ ว่า `alpha` และคืนคะแนนเป็นค่าลบของความคลาดเคลื่อน จึงต้องกลับเครื่องหมาย

### โค้ดที่ 5.5 — การตรวจสอบไขว้ห้าส่วนเพื่อหา λ ที่เหมาะสม

In [ ]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import Ridge

lambdas = np.logspace(-4, 2, 20)
kfold = KFold(n_splits=5, shuffle=True, random_state=0)
cv_scores = []
for lam in lambdas:
    pipe = Pipeline([
        ('scale_x', StandardScaler()),
        ('poly',    PolynomialFeatures(degree=10, include_bias=False)),
        ('scale_f', StandardScaler()),
        ('reg',     Ridge(alpha=lam)),
    ])
    scores = cross_val_score(pipe, X_an, F_an, cv=kfold,
                             scoring='neg_mean_squared_error')
    cv_scores.append(-scores.mean())
optimal_lambda = lambdas[np.argmin(cv_scores)]

In [ ]:
print(f"optimal lambda = {optimal_lambda:.3f}, CV MSE = {min(cv_scores):.4f} N^2 (sigma^2 = 0.0025)")
flat = lambdas[np.array(cv_scores) < 1.1 * min(cv_scores)]
print(f"lambdas within 10 % of the minimum: {flat.min():.3f} to {flat.max():.3f}  (a flat valley)")

plt.figure(figsize=(6.5, 3.6))
plt.semilogx(lambdas, cv_scores, "o-"); plt.axvline(optimal_lambda, ls=":")
plt.axhline(0.0025, ls="--", color="k", label=r"$\sigma^2$")
plt.xlabel(r"$\lambda$ (alpha)"); plt.ylabel(r"5-fold CV MSE (N$^2$)"); plt.legend()
plt.title("ridge penalty for a degree-10 polynomial (SYNTHETIC)"); plt.tight_layout(); plt.show()

เส้นโค้งเป็นรูปตัวยู คะแนนต่ำสุดอยู่ที่ $\lambda \approx 0.3$ ด้วยความคลาดเคลื่อนราว 0.0030 N$^2$ ใกล้พื้นขั้นต่ำ $\sigma^2$ เช่นเดียวกับพหุนามดีกรี 3 ที่ $\lambda$ เล็กมาก แบบจำลองเรียนรู้เกิน ที่ $\lambda$ ใหญ่มาก สัมประสิทธิ์ถูกบีบจนแบบจำลองแข็งเกินไป และหุบรอบจุดต่ำสุดค่อนข้างแบน การรายงาน $\lambda^*$ ละเอียดหลายหลักจึงไม่มีความหมาย

ด้านซ้ายของเส้นโค้งไม่ได้สูงขึ้นอย่างเรียบ แต่มีโหนกที่ $\lambda$ ราว $4\times10^{-4}$ เซลล์ถัดไปดูความคลาดเคลื่อนของแต่ละส่วนเพื่อหาสาเหตุ

In [ ]:
print("per-fold CV MSE at small lambda (and lambda = 0, plain OLS of degree 10):")
for lam in (0.0, 1e-4, 4.3e-4, 0.3):
    reg = LinearRegression() if lam == 0 else Ridge(alpha=lam)
    s_ = -cross_val_score(poly10(reg), X_an, F_an, cv=kfold, scoring='neg_mean_squared_error')
    print(f"  lambda = {lam:<7g} folds {np.round(s_, 4)}  mean {s_.mean():.4f}")
tr, va = next(iter(kfold.split(X_an)))
print(f"fold 1: validation x reaches {x_an[va].max():.3f} m, training x only up to {x_an[tr].max():.3f} m")

ส่วนแรกมีจุดตรวจสอบที่ $x = 0.140$ m ขณะที่ข้อมูลฝึกของรอบนั้นไปถึงเพียง 0.122 m พหุนามดีกรี 10 ที่ถูกลงโทษน้อยจึงต้องประมาณนอกช่วง (extrapolation) และแกว่งแรง ความคลาดเคลื่อนของส่วนนี้ส่วนเดียวครอบงำค่าเฉลี่ย

> **ความเข้าใจคลาดเคลื่อนที่พบบ่อย:** "การแบ่งส่วนของการตรวจสอบไขว้ไม่จำเป็นต้องสุ่ม" ข้อมูลในห้องปฏิบัติการมักบันทึกเรียงตามตัวแปรที่ควบคุม ถ้าแบ่งตามลำดับโดยไม่สลับ ส่วนแรกจะมีแต่ระยะยืดน้อย และแบบจำลองต้องประมาณนอกช่วง จึงควรใช้ `shuffle=True` ข้อยกเว้นคือข้อมูลอนุกรมเวลา ซึ่งต้องแบ่งตามเวลา (บทที่ 9)

### เกณฑ์สารสนเทศ
เกณฑ์ของอะกะอิเกะ $\mathrm{AIC} = 2p - 2\ln\hat L$ และเกณฑ์ของเบส์ $\mathrm{BIC} = p\ln N - 2\ln\hat L$ ให้คะแนนความเข้ากับข้อมูลหักด้วยค่าปรับของความซับซ้อน ค่าต่ำกว่าถือว่าดีกว่า ไม่ต้องแบ่งข้อมูล แต่ต้องมีแบบจำลองความน่าจะเป็นที่ชัดเจน (รายละเอียดในบทที่ 9) เซลล์ถัดไปคำนวณทั้งสองค่าให้พหุนามดีกรี 1–8 ด้วย statsmodels

In [ ]:
z_an = (x_an - x_an.mean()) / x_an.std()
ic = []
for d in range(1, 9):
    f_ = sm.OLS(F_an, sm.add_constant(np.column_stack([z_an**j for j in range(1, d + 1)]))).fit()
    ic.append((d, f_.aic, f_.bic))
ic = pd.DataFrame(ic, columns=["degree", "AIC", "BIC"]).set_index("degree")
print(ic.round(1).to_string())
print("lowest AIC at degree", ic["AIC"].idxmin(), "| lowest BIC at degree", ic["BIC"].idxmin())

ทั้งสองเกณฑ์เลือกดีกรี 3 เช่นเดียวกับการตรวจสอบไขว้ BIC ลงโทษพารามิเตอร์เพิ่มแรงกว่า AIC เมื่อ $N$ มาก ($\ln 60 \approx 4.1 > 2$)

### λ ในมุมมองของฟิสิกส์
เมื่อ $\lambda = 0$ ไม่มีข้อจำกัด เมื่อ $\lambda\to\infty$ แบบจำลองเหลือเพียงค่าคงที่ ในมุมมองแบบเบส์ (บทที่ 9) ริดจ์เท่ากับการสมมติล่วงหน้าว่าสัมประสิทธิ์แจกแจงแบบเกาส์รอบศูนย์ด้วยความแปรปรวน $\tau^2$ และ $\lambda = \sigma^2/\tau^2$ ข้อมูลที่มีสัญญาณรบกวนมากจึงต้องการ $\lambda$ ใหญ่ขึ้น ส่วนแลสโซสอดคล้องกับความเชื่อว่าสัมประสิทธิ์ส่วนใหญ่เป็นศูนย์ เหมาะกับระบบที่มีโครงสร้างเบาบาง

> **ความเข้าใจคลาดเคลื่อนที่พบบ่อย:** "$\lambda$ ที่ดีที่สุดคือระดับสัญญาณรบกวนของระบบ" $\lambda$ ขึ้นกับอัตราส่วนระหว่างสัญญาณรบกวนกับขนาดของสัมประสิทธิ์ วิธีปรับมาตราส่วน จำนวนข้อมูล และวิธีที่ไลบรารีเขียนฟังก์ชันเป้าหมาย ค่า $\lambda \approx 0.3$ จึงไม่ใช่ปริมาณทางฟิสิกส์ที่ย้ายไปใช้กับการทดลองอื่นได้

แบบฝึกหัด E5.4 (เลือก $\lambda$ เมื่อสัญญาณรบกวนเพิ่มเป็น 0.15 N) เป็นงานนอกชั้นในส่วนที่ 18

---
# ส่วนที่ 12 การปรับเส้นโค้งด้วยไคกำลังสอง: ออสซิลเลเตอร์แบบหน่วง (หัวข้อ 5.4.1–5.4.2)

บทที่ 4 ใช้ไคกำลังสองทดสอบว่าข้อมูลเข้ากับแบบจำลองที่กำหนดหรือไม่ หัวข้อนี้ใช้ปริมาณเดียวกันในทางกลับกัน คือหาพารามิเตอร์ที่ทำให้แบบจำลองเข้ากับข้อมูลได้ดีที่สุด

$$\chi^2(\boldsymbol\theta) = \sum_{i=1}^{N}\left(\frac{y_i - f(x_i;\boldsymbol\theta)}{\sigma_i}\right)^2$$

แต่ละจุดถูกถ่วงน้ำหนักด้วย $1/\sigma_i^2$ ถ้า $\sigma_i$ เท่ากันทุกจุด การทำให้ $\chi^2$ น้อยที่สุดก็คือ OLS การปรับด้วยไคกำลังสองจึงเป็นรูปทั่วไปของ OLS ข้อที่ต่างคือ $f$ ไม่จำเป็นต้องเป็นเชิงเส้นในพารามิเตอร์ จึงต้องหาค่าต่ำสุดเชิงตัวเลข และใส่ความรู้ล่วงหน้าทางฟิสิกส์ (physical prior) เป็นขอบเขตของพารามิเตอร์ได้โดยตรง

ออสซิลเลเตอร์แบบหน่วงน้อยมีการกระจัด $x(t) = A\,e^{-\gamma t}\cos(\omega t + \phi)$ ซึ่งไม่เป็นเชิงเส้นใน $\gamma$ และ $\omega$ โค้ดที่ 5.6 สร้างข้อมูลที่ $A = 1.0$ m, $\gamma = 0.3$ s$^{-1}$, $\omega = 6.0$ rad/s, $\phi = 0.5$ rad วัด 200 จุดใน 10 s ด้วยสัญญาณรบกวน 0.05 m แล้วปรับด้วย `scipy.optimize.curve_fit` ซึ่งใช้วิธีบริเวณที่เชื่อถือได้ (trust region reflective) เมื่อมีขอบเขต

### โค้ดที่ 5.6 — การปรับเส้นโค้งของออสซิลเลเตอร์แบบหน่วงพร้อมขอบเขตทางฟิสิกส์

In [ ]:
from scipy.optimize import curve_fit

def damped_oscillator(t, A, gamma, omega, phi):
    return A * np.exp(-gamma * t) * np.cos(omega * t + phi)

rng = np.random.default_rng(seed=2026)
t = np.linspace(0, 10, 200)                               # s
sigma_x = 0.05                                            # m
x_obs = damped_oscillator(t, 1.0, 0.3, 6.0, 0.5) \
        + rng.normal(scale=sigma_x, size=t.size)

p0 = [1.0, 0.2, 6.0, 0.0]                                 # initial guess
bounds = ([0, 0, 0, -np.pi], [10, 5, 20, np.pi])          # physical limits
popt, pcov = curve_fit(damped_oscillator, t, x_obs,
                       p0=p0, bounds=bounds,
                       sigma=sigma_x * np.ones(t.size),
                       absolute_sigma=True)

A, gamma, omega, phi = popt
errs = np.sqrt(np.diag(pcov))
print(f"gamma = {gamma:.4f} +/- {errs[1]:.4f} 1/s")
print(f"omega = {omega:.4f} +/- {errs[2]:.4f} rad/s")

In [ ]:
print(f"A = {A:.4f} +/- {errs[0]:.4f} m, phi = {phi:.4f} +/- {errs[3]:.4f} rad")
print(f"omega is {(omega - 6.0) / errs[2]:+.1f} sigma from 6.0, gamma is {(gamma - 0.3) / errs[1]:+.1f} sigma from 0.3")
if not os.path.exists("data/damped_oscillator.csv"):     # same data file as the companion code (used again in chapter 9)
    pd.DataFrame({"t_s": t, "x_m": x_obs}).to_csv("data/damped_oscillator.csv", index=False)

resid_d = x_obs - damped_oscillator(t, *popt)
fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
a1.plot(t, x_obs, ".", ms=3, label="data"); a1.plot(t, damped_oscillator(t, *popt), label="fit")
a1.plot(t, A * np.exp(-gamma * t), "k:", lw=1, label=r"envelope $A e^{-\gamma t}$")
a1.set_ylabel("x (m)"); a1.legend(fontsize=8); a1.set_title("damped oscillator (SYNTHETIC)")
a2.plot(t, resid_d / sigma_x, ".", ms=3); a2.axhline(0, color="k", lw=0.8)
a2.set_xlabel("t (s)"); a2.set_ylabel(r"residual / $\sigma$")
plt.tight_layout(); plt.show()

ผลคือ $\gamma = (0.286 \pm 0.008)$ s$^{-1}$ และ $\omega = (6.010 \pm 0.007)$ rad/s ห่างจากค่าจริงราว $1.9\sigma$ และ $1.3\sigma$ ซึ่งอยู่ในขอบเขตที่คาดได้จากสัญญาณรบกวน

### บทบาทของอาร์กิวเมนต์ใน `curve_fit`
| อาร์กิวเมนต์ | บทบาท |
|---|---|
| `p0` | ค่าเริ่มต้นของการค้นหา ควรประมาณจากกราฟของข้อมูล |
| `bounds` | ขอบเขตทางฟิสิกส์ของพารามิเตอร์ |
| `sigma` | ความไม่แน่นอนของแต่ละจุด |
| `absolute_sigma=True` | บอกว่า `sigma` เป็นความไม่แน่นอนจริงในหน่วยของข้อมูล ถ้าไม่ระบุ ฟังก์ชันจะถือว่าเป็นเพียงน้ำหนักสัมพัทธ์ แล้วปรับขนาด `pcov` ด้วย $\chi^2_{\text{red}}$ |
| `popt`, `pcov` | พารามิเตอร์ที่ดีที่สุด และเมทริกซ์ความแปรปรวนร่วม ซึ่งสมาชิกนอกเส้นทแยงบอกว่าพารามิเตอร์คู่ใดสัมพันธ์กัน |

เซลล์ถัดไปแสดงผลของ `absolute_sigma` ด้วยตัวเลข

In [ ]:
_, pcov_rel = curve_fit(damped_oscillator, t, x_obs, p0=p0, bounds=bounds, sigma=sigma_x * np.ones(t.size))
chi2_red_d = np.sum((resid_d / sigma_x)**2) / (t.size - 4)
print(f"SE with absolute_sigma=True : {np.round(errs, 4)}")
print(f"SE with absolute_sigma=False: {np.round(np.sqrt(np.diag(pcov_rel)), 4)}")
print(f"ratio = {np.sqrt(np.diag(pcov_rel))[1] / errs[1]:.3f} = sqrt(chi2_red) = {np.sqrt(chi2_red_d):.3f}")

เมื่อรู้ความไม่แน่นอนของการวัดจริง ควรใช้ `absolute_sigma=True` เพื่อให้ความคลาดเคลื่อนของพารามิเตอร์สะท้อน $\sigma$ ที่วัดได้ และใช้ $\chi^2_{\text{red}}$ ตรวจแบบจำลองแยกต่างหาก (ส่วนที่ 16)

---
# ส่วนที่ 13 ค่าเริ่มต้นจากกราฟและตัวประกอบคุณภาพ (หัวข้อ 5.4.2, 5.4.5)

การปรับเส้นโค้งไม่เชิงเส้นไม่มีการรับประกันว่าจะพบค่าต่ำสุดที่แท้จริง ฟังก์ชัน $\chi^2$ ของแบบจำลองการแกว่งอาจมีหุบย่อยหลายหุบ ถ้าค่าเริ่มต้นไกลเกินไป การค้นหาอาจติดอยู่ในหุบย่อย เซลล์ถัดไปลองค่าเริ่มต้นของ $\omega$ ตั้งแต่ 1 ถึง 18 rad/s

In [ ]:
same = []
for w0 in np.arange(1, 19):
    try:
        p_, _ = curve_fit(damped_oscillator, t, x_obs, p0=[1.0, 0.2, w0, 0.0], bounds=bounds,
                          sigma=sigma_x * np.ones(t.size), absolute_sigma=True)
        same.append(abs(p_[2] - omega) < 1e-3)
    except RuntimeError:
        same.append(False)
print(f"starting omega 1..18 rad/s: converged to omega = {omega:.3f} from {sum(same)} of 18 starts")

# chi^2 as a function of omega alone (A, gamma, phi fixed at the best fit): the landscape the search must cross
wgrid = np.linspace(1, 18, 800)
chi2_w = [np.sum(((x_obs - damped_oscillator(t, A, gamma, w, phi)) / sigma_x)**2) for w in wgrid]
plt.figure(figsize=(7, 3)); plt.semilogy(wgrid, chi2_w); plt.axvline(omega, color="r", ls=":")
plt.xlabel(r"$\omega$ (rad/s)"); plt.ylabel(r"$\chi^2$"); plt.title(r"$\chi^2$ along $\omega$ with other parameters fixed")
plt.tight_layout(); plt.show()

ในตัวอย่างนี้ ซึ่งข้อมูลมีหลายรอบและสัญญาณรบกวนต่ำ การค้นหาลู่เข้าสู่ค่าเดียวกันจากทุกค่าเริ่มต้น แต่ภาพตัดของ $\chi^2$ ตามแกน $\omega$ เตือนว่าเรื่องนี้ไม่ได้เกิดขึ้นเสมอ หุบของค่าต่ำสุดแคบมาก และเมื่อ $\omega$ ห่างจากค่าจริงเกินราว 1 rad/s ค่า $\chi^2$ แทบคงที่ ข้อมูลจึงแทบไม่บอกทิศทางแก่การค้นหา ที่การค้นหาในตัวอย่างนี้ยังลู่เข้าได้ เพราะอัลกอริทึมปรับพารามิเตอร์ทั้งสี่ตัวพร้อมกัน ไม่ได้เดินตามภาพตัดนี้ แต่กับข้อมูลที่สั้นกว่าหรือมีสัญญาณรบกวนมากกว่า การค้นหาอาจหยุดที่ค่าต่ำสุดย่อย เช่น ที่ความถี่ครึ่งหนึ่งหรือสองเท่าของความถี่จริง หลักปฏิบัติคือประมาณค่าเริ่มต้นจากกราฟเสมอ และลองค่าเริ่มต้นหลายค่า

### ✏️ แบบฝึก E5.5 (ส่วนในชั้น 15 นาที) การปรับเส้นโค้งของออสซิลเลเตอร์แบบหน่วง
1. ดูกราฟในเซลล์ถัดไป แล้วประมาณค่าเริ่มต้นเอง
   - $A$: อ่านจากยอดแรก
   - $\omega$: นับจำนวนรอบในช่วงเวลาหนึ่ง $\omega \approx 2\pi \times (\text{จำนวนรอบ})/\Delta t$
   - $\gamma$: จากยอดสองยอดที่ห่างกัน $\Delta t$ ตาม $\gamma \approx \ln(A_1/A_2)/\Delta t$
2. ปรับเส้นโค้งด้วยค่าเริ่มต้นของตนเอง รายงาน $\gamma$ และ $\omega$ พร้อมความคลาดเคลื่อนมาตรฐานและหน่วย
3. คำนวณ $Q = \omega/(2\gamma)$ (ใช้ $\omega$ แทน $\omega_0$ ได้เพราะ $\gamma/\omega \approx 0.05$) เก็บใน `my_Q` แล้วเทียบกับค่าจริง $Q = \omega_0/(2\gamma) = 10.01$
4. ดูกราฟส่วนเหลือเทียบกับเวลา ว่ามีรูปแบบหรือไม่

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(t, x_obs, ".-", ms=3, lw=0.5)
ax.set_xticks(np.arange(0, 10.5, 0.5), minor=True); ax.grid(which="minor", alpha=0.2)
ax.set_xlabel("t (s)"); ax.set_ylabel("x (m)"); ax.set_title("read A, omega and gamma from this graph (SYNTHETIC)")
plt.tight_layout(); plt.show()

In [ ]:
my_A0 = None         # ✏️ m
my_omega0 = None     # ✏️ rad/s
my_gamma0 = None     # ✏️ 1/s

In [ ]:
# fit with your own starting values (phi starts at 0)
if None in (my_A0, my_omega0, my_gamma0):
    print("(ยังไม่ได้เติมค่าเริ่มต้น) เติมค่าในเซลล์ก่อนหน้าแล้วรันเซลล์นี้อีกครั้ง")
    popt5 = pcov5 = None
else:
    popt5, pcov5 = curve_fit(damped_oscillator, t, x_obs, p0=[my_A0, my_gamma0, my_omega0, 0.0],
                             bounds=bounds, sigma=sigma_x * np.ones(t.size), absolute_sigma=True)
    e5 = np.sqrt(np.diag(pcov5))
    print(f"gamma = ({popt5[1]:.3f} +/- {e5[1]:.3f}) 1/s, omega = ({popt5[2]:.3f} +/- {e5[2]:.3f}) rad/s")
    print(f"relative uncertainty: gamma {100 * e5[1] / popt5[1]:.1f} %, omega {100 * e5[2] / popt5[2]:.2f} %")
    r5 = x_obs - damped_oscillator(t, *popt5)
    plt.figure(figsize=(8, 2.6)); plt.plot(t, r5, ".", ms=3); plt.axhline(0, color="k", lw=0.8)
    plt.xlabel("t (s)"); plt.ylabel("residual (m)"); plt.title("residuals of your fit"); plt.tight_layout(); plt.show()

In [ ]:
my_Q = None      # ✏️ Q = omega / (2 gamma) from popt5

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_A0', 'my_Q', 'my_gamma0', 'my_omega0'])
_guess = {"A": (my_A0, 0.96), "omega": (my_omega0, 6.01), "gamma": (my_gamma0, 0.29)}
for k_, (mine, ref) in _guess.items():
    _say(f"ค่าเริ่มต้น {k_:5s}:", "(ยังไม่ได้ตอบ)" if mine is None else
          ("✔ อยู่ในช่วงที่ดี" if abs(mine / ref - 1) < 0.35 else f"△ ห่างจากผลการปรับ ({ref}) เกินร้อยละ 35 ลองอ่านกราฟใหม่"))
if popt5 is not None:
    _say("ผลการปรับ:", "✔ ลู่เข้าสู่ค่าต่ำสุดเดียวกับโค้ดที่ 5.6" if abs(popt5[2] - omega) < 1e-3
          else "✘ ติดอยู่ในหุบย่อย ลองค่าเริ่มต้นใหม่")
_say("Q:", "(ยังไม่ได้ตอบ)" if my_Q is None else
      ("✔" if abs(my_Q - omega / (2 * gamma)) < 0.05 else f"✘ ค่าที่คาดคือ {omega / (2 * gamma):.2f}"),
      f"— Q = {omega / (2 * gamma):.1f} ต่างจากค่าจริง 10.0 ราว 1.8 sigma (sigma_Q มาจากส่วนที่ 17)")

In [ ]:
#@markdown แนวคำตอบ E5.5: ค่าเริ่มต้นจากยอดของกราฟด้วย find_peaks (รันหลังตอบเองแล้ว)
_say = _gate(['my_A0', 'my_Q', 'my_gamma0', 'my_omega0'])
from scipy.signal import find_peaks
pk, _ = find_peaks(x_obs, distance=15)
w_guess = 2 * np.pi * (len(pk) - 1) / (t[pk[-1]] - t[pk[0]])
g_guess = np.log(x_obs[pk[0]] / x_obs[pk[4]]) / (t[pk[4]] - t[pk[0]])
_say(f"{len(pk)} maxima; omega from their spacing = {w_guess:.2f} rad/s; "
      f"A from the first maximum = {x_obs[pk[0]]:.2f} m; gamma from maxima 1 and 5 = {g_guess:.2f} 1/s")
_say(f"fit: gamma = ({gamma:.3f} +/- {errs[1]:.3f}) 1/s, omega = ({omega:.3f} +/- {errs[2]:.3f}) rad/s, Q = {omega / (2 * gamma):.1f}")
_say(f"residuals: mean {resid_d.mean():+.4f} m, std {resid_d.std(ddof=4):.4f} m (noise 0.05 m), "
      f"lag-1 autocorrelation {np.corrcoef(resid_d[:-1], resid_d[1:])[0, 1]:+.2f} -> no pattern")
_say('''ความคลาดเคลื่อนสัมพัทธ์ของ omega (0.12 %) เล็กกว่าของ gamma (2.7 %) มาก เพราะการแกว่งหลายรอบกำหนดความถี่ได้แม่นยำ
ขณะที่การหน่วงต้องประเมินจากการเปลี่ยนแปลงของแอมพลิจูด ซึ่งถูกสัญญาณรบกวนกลบมากกว่า
A จากยอดแรกได้ราว 0.71 m ต่ำกว่า A = 0.96 m ที่ปรับได้ เพราะยอดแรกไม่ได้อยู่ที่ t = 0 และเฟสไม่เป็นศูนย์ แต่ยังพอเป็นค่าเริ่มต้น''')

---
# คาบที่ 4 ขอบเขตทางฟิสิกส์ การประเมินแบบจำลอง และการแพร่ของความไม่แน่นอน

> **เริ่มคาบนี้หลังจากปิด Colab ไปแล้ว:** คลิกเซลล์นี้ แล้วเลือก Runtime → Run before เพื่อสร้างข้อมูลและตัวแปรจากคาบก่อนหน้าใหม่ก่อนเรียนต่อ (คาบนี้ใช้ผลของโค้ดที่ 5.6)

---
# ส่วนที่ 14 ขอบเขตทางฟิสิกส์และสมการของช็อกลีย์ (หัวข้อ 5.4.3–5.4.5)

ขอบเขตในโค้ดที่ 5.6 ไม่ได้ตั้งขึ้นเพื่อช่วยให้การคำนวณลู่เข้าเพียงอย่างเดียว แต่ละขอบเขตมีเหตุผลทางฟิสิกส์รองรับ

| ขอบเขต | เหตุผลทางฟิสิกส์ |
|---|---|
| $\gamma \ge 0$ | ระบบที่ไม่มีแหล่งพลังงานภายนอกสูญเสียพลังงานได้อย่างเดียว |
| $\omega > 0$ | นิยามของความถี่เชิงมุม เครื่องหมายของการแกว่งรวมอยู่ในเฟสแล้ว |
| $T > 0$ K | อุณหภูมิสัมบูรณ์ |
| $E = h\nu \ge 0$ | พลังงานของโฟตอน |
| $\lvert\vec v\rvert < c$ | ทฤษฎีสัมพัทธภาพพิเศษ สำหรับอนุภาคที่มีมวล |
| $1 \le n \le 2$ | ตัวประกอบอุดมคติของไดโอดตามแบบจำลองการแพร่และการรวมตัวใหม่ |

ขอบเขตบังคับเพียงให้ผลอยู่ในบริเวณที่เป็นไปได้ทางฟิสิกส์ ไม่ได้บังคับให้ผลออกมาตามที่อยากได้ เซลล์ถัดไป (ตัวอย่างเพิ่มเติมจากตำรา) สร้างข้อมูลสั้น 3 s ของออสซิลเลเตอร์ที่ถูกขับจนแอมพลิจูด **เพิ่มขึ้น** ($\gamma = -0.08$ s$^{-1}$) แล้วปรับด้วยขอบเขต $\gamma \ge 0$ ชุดเดิม

In [ ]:
rng_b = np.random.default_rng(1)
t_b = np.linspace(0, 3, 60)
x_b = damped_oscillator(t_b, 1.0, -0.08, 6.0, 0.5) + rng_b.normal(scale=0.05, size=t_b.size)
p_b, c_b = curve_fit(damped_oscillator, t_b, x_b, p0=[1.0, 0.2, 6.0, 0.0], bounds=bounds,
                     sigma=0.05 * np.ones(t_b.size), absolute_sigma=True)
p_u, c_u = curve_fit(damped_oscillator, t_b, x_b, p0=[1.0, 0.2, 6.0, 0.0],
                     sigma=0.05 * np.ones(t_b.size), absolute_sigma=True)
print(f"with gamma >= 0 : gamma = {p_b[1]:.4f} +/- {np.sqrt(c_b[1, 1]):.4f} 1/s   <- sits on the bound")
print(f"without bounds  : gamma = {p_u[1]:.3f} +/- {np.sqrt(c_u[1, 1]):.3f} 1/s   <- the data want a growing amplitude")

> **ความเข้าใจคลาดเคลื่อนที่พบบ่อย:** "ถ้าใส่ขอบเขตแล้ว ผลที่ได้ย่อมถูกต้องทางฟิสิกส์" พารามิเตอร์ที่ชนขอบเขตพอดี เช่น $\gamma = 0$ หรือ $n = 2.000$ เป็นสัญญาณเตือน ไม่ใช่ผลลัพธ์ ค่าที่ชนขอบบอกว่าข้อมูล "อยาก" ไปอยู่นอกบริเวณที่เป็นไปได้ สาเหตุอาจเป็นแบบจำลองที่ขาดกลไก (ในตัวอย่างนี้คือแรงขับภายนอก) ค่าเริ่มต้นที่ไม่ดี หรือข้อมูลที่มีปัญหา และความคลาดเคลื่อนมาตรฐานที่รายงานสำหรับพารามิเตอร์ที่ชนขอบเขตไม่มีความหมาย เพราะสูตรสมมติว่าค่าต่ำสุดอยู่ภายในบริเวณ

### ✏️ แบบฝึก 5.H พารามิเตอร์ชนขอบเขต (คำถามทบทวนที่ 5)
การปรับออสซิลเลเตอร์ด้วยขอบเขต $\gamma \ge 0$ ให้ $\gamma = 0.000$ พอดี เติม `True` หรือ `False` ให้แต่ละข้อความ

In [ ]:
my_bound = {
    1: None,   # ✏️ ควรรายงานว่าระบบไม่มีการหน่วง gamma = 0.000 +/- (SE จาก pcov)
    2: None,   # ✏️ สาเหตุหนึ่งที่เป็นไปได้คือแอมพลิจูดเพิ่มขึ้นจริงจากแรงขับภายนอก
    3: None,   # ✏️ สาเหตุหนึ่งที่เป็นไปได้คือช่วงเวลาที่วัดสั้นเกินกว่าจะเห็นการหน่วง
    4: None,   # ✏️ ถ้าข้อมูลเข้ากับการไม่มีการหน่วงจริง สิ่งที่รายงานได้คือขีดจำกัดบน เช่น gamma < 0.01 1/s
}

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_bound'])
_bd = {1: (False, "ค่าที่ชนขอบเขตเป็นสัญญาณเตือน และ SE ของค่านั้นใช้ไม่ได้"),
       2: (True, "ตรงกับตัวอย่างในเซลล์ก่อนหน้า"),
       3: (True, "การหน่วงเล็กมากเมื่อเทียบกับช่วงเวลาที่วัด"),
       4: (True, "รายงานเป็นขีดจำกัดบนที่ระดับความเชื่อมั่นหนึ่ง")}
for k_, (v, why) in _bd.items():
    mine = my_bound[k_]
    _say(f"ข้อ {k_}:", "(ยังไม่ได้ตอบ)" if mine is None else ("✔" if mine == v else "✘"), "—", why)

### ตัวอย่างทางไฟฟ้า: สมการของช็อกลีย์
กระแสผ่านไดโอดรอยต่อพีเอ็นเมื่อได้รับไบแอสตรงคือ $I = I_0\left(e^{V/(nV_T)} - 1\right)$ เมื่อ $I_0$ คือกระแสอิ่มตัวย้อนกลับ (ระดับพิโกแอมแปร์สำหรับไดโอดซิลิคอนเล็ก ๆ) $n$ คือตัวประกอบอุดมคติ (ideality factor) และ $V_T = k_BT/q \approx 25.85$ mV ที่ 300 K โค้ดที่ 5.7 ปรับสมการนี้กับข้อมูลจำลองที่ $I_0 = 2$ pA, $n = 1.4$ วัดที่ 0.30–0.65 V ด้วยสัญญาณรบกวนร้อยละ 5 ของกระแส ผลลัพธ์ตั้งชื่อ `popt_d`, `pcov_d` เพื่อไม่ให้ทับผลของออสซิลเลเตอร์ที่ส่วนที่ 17 ใช้ต่อ

### โค้ดที่ 5.7 — การปรับสมการของช็อกลีย์พร้อมขอบเขตทางฟิสิกส์

In [ ]:
V_T = 0.02585                                   # V, thermal voltage at 300 K

def shockley(V, I0, n):
    return I0 * (np.exp(V / (n * V_T)) - 1)

rng = np.random.default_rng(seed=2026)
V = np.linspace(0.30, 0.65, 36)                 # V
I_obs = shockley(V, 2e-12, 1.4) * (1 + rng.normal(scale=0.05, size=V.size))

p0 = [1e-12, 1.5]
bounds = ([1e-15, 1.0], [1e-9, 2.0])            # I0 in A; n from device physics
popt_d, pcov_d = curve_fit(shockley, V, I_obs, p0=p0, bounds=bounds,
                           sigma=0.05 * I_obs, absolute_sigma=True)

In [ ]:
e_d = np.sqrt(np.diag(pcov_d))
rho_d = pcov_d[0, 1] / (e_d[0] * e_d[1])
print(f"I0 = ({popt_d[0] * 1e12:.2f} +/- {e_d[0] * 1e12:.2f}) pA, n = {popt_d[1]:.3f} +/- {e_d[1]:.3f}")
print(f"correlation(I0, n) = {rho_d:.3f}")
print(f"V_T = k_B T / q at 300 K = {1.380649e-23 * 300 / 1.602176634e-19 * 1e3:.2f} mV")
print(f"current range: {I_obs.min():.2e} A to {I_obs.max():.2e} A  (more than four decades)")
pu, _ = curve_fit(shockley, V, I_obs, p0=p0, bounds=bounds)       # no sigma: every point weighted equally
print(f"unweighted fit for comparison: I0 = {pu[0] * 1e12:.2f} pA, n = {pu[1]:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].semilogy(V, I_obs, "o", ms=4, label="data")
ax[0].semilogy(V, shockley(V, *popt_d), label="weighted fit")
ax[0].semilogy(V, shockley(V, *pu), "--", label="unweighted fit")
ax[0].set_xlabel("V (V)"); ax[0].set_ylabel("I (A)"); ax[0].legend(fontsize=8); ax[0].set_title("diode I-V (SYNTHETIC)")
dI, dn = np.linspace(-3, 3, 61), np.linspace(-3, 3, 61)
G = np.array([[np.sum(((I_obs - shockley(V, popt_d[0] + a * e_d[0], popt_d[1] + b * e_d[1])) / (0.05 * I_obs))**2)
               for a in dI] for b in dn])
cs = ax[1].contour(dI, dn, G - G.min(), levels=[1, 4, 9]); ax[1].clabel(cs, fmt={1: "1", 4: "4", 9: "9"})
ax[1].set_xlabel(r"$(I_0 - \hat I_0)/\sigma_{I_0}$"); ax[1].set_ylabel(r"$(n - \hat n)/\sigma_n$")
ax[1].set_title(r"$\Delta\chi^2$ contours: a narrow tilted valley"); plt.tight_layout(); plt.show()

ผลคือ $I_0 = (1.94 \pm 0.08)$ pA และ $n = 1.397 \pm 0.004$ ค่า $n$ ใกล้ 1 บอกว่ากระแสส่วนใหญ่มาจากการแพร่ของพาหะข้ามรอยต่อ ค่าใกล้ 2 บอกว่ามาจากการรวมตัวใหม่ในบริเวณปลอดพาหะ ค่า 1.4 จึงบอกว่ามีทั้งสองกลไกปนกัน

สังเกตสองจุด จุดแรก `sigma=0.05*I_obs` บอกว่าความไม่แน่นอนเป็นสัดส่วนกับกระแส ข้อมูลจึงมีความแปรปรวนไม่คงที่ การปรับแบบไม่ถ่วงน้ำหนักให้น้ำหนักกับจุดที่แรงดันสูงมากเกินไป และได้ $n \approx 1.72$ ซึ่งผิดไปมาก จุดที่สอง สหสัมพันธ์ระหว่าง $I_0$ กับ $n$ สูงถึง 0.98 เพราะในช่วงที่วัด เส้นโค้งเกือบเป็นเส้นตรงบนกราฟลอการิทึม การเพิ่ม $n$ เล็กน้อยทำให้เส้นชันน้อยลง ซึ่งชดเชยได้ด้วยการเพิ่ม $I_0$ ภาพ $\Delta\chi^2$ ทางขวาจึงเป็นหุบแคบที่เอียง

### การตีความพารามิเตอร์ของออสซิลเลเตอร์
| พารามิเตอร์ | หน่วย | ความหมาย | ปริมาณที่คำนวณต่อได้ |
|---|---|---|---|
| $\gamma$ | s$^{-1}$ | อัตราการลดลงของแอมพลิจูด | เวลา $1/\gamma$ และตัวประกอบคุณภาพ $Q$ |
| $\omega$ | rad/s | ความถี่เชิงมุมที่สังเกตได้ | คาบ $2\pi/\omega$ และ $\omega_0 = \sqrt{\omega^2+\gamma^2}$ |
| $A$ | m | แอมพลิจูดเริ่มต้น | พลังงานเริ่มต้น $E_0 \approx \tfrac12 m\omega_0^2A^2$ |
| $\phi$ | rad | เฟสเริ่มต้น | ขึ้นกับการเลือกจุดเริ่มนับเวลา ไม่มีความหมายทางฟิสิกส์ของระบบ |

ตัวประกอบคุณภาพ (quality factor) $Q = \omega_0/(2\gamma)$ บอกโดยประมาณว่าระบบแกว่งได้กี่เรเดียนก่อนพลังงานลดลงเหลือ $1/e$ ระบบที่ $Q > 1/2$ หน่วงน้อยและแกว่งได้ $Q = 1/2$ หน่วงวิกฤต และ $Q < 1/2$ หน่วงมาก ซึ่งแบบจำลองโคไซน์ในโค้ดที่ 5.6 ใช้ไม่ได้

In [ ]:
omega0 = np.sqrt(omega**2 + gamma**2)
print(f"Q = omega0/(2 gamma) = {omega0 / (2 * gamma):.2f} (true {np.sqrt(36 + 0.09) / 0.6:.2f}) -> underdamped (Q > 1/2)")
print(f"1/gamma = {1 / gamma:.2f} s; oscillations within 1/gamma: {omega / gamma / (2 * np.pi):.1f}")
print(f"gamma/omega = {gamma / omega:.3f}; (omega0 - omega)/omega = {100 * (omega0 / omega - 1):.2f} % -> omega can replace omega0 here")

ระบบที่ $Q$ สูงมาก เช่น ส้อมเสียงหรือผลึกควอตซ์ในนาฬิกาซึ่งมี $Q$ ระดับหมื่นถึงล้าน เป็นหัวใจของอุปกรณ์รักษาความถี่ ข้อสรุปของหัวข้อนี้คือ พารามิเตอร์ที่ได้จากการปรับเส้นโค้งต้องได้รับการปฏิบัติเหมือนผลการวัดทางฟิสิกส์ทุกประการ คือมีหน่วย เปรียบเทียบกับทฤษฎีได้ มีความไม่แน่นอนและสหสัมพันธ์กำกับ และขอบเขตที่ใส่ต้องมีเหตุผลทางฟิสิกส์

---
# ส่วนที่ 15 การวิเคราะห์ส่วนเหลือ (หัวข้อ 5.5.1)

การปรับเส้นโค้งให้ตัวเลขเสมอ ไม่ว่าแบบจำลองจะถูกหรือผิด ส่วนเหลือ $r_i = y_i - \hat y_i$ คือสิ่งที่แบบจำลองอธิบายไม่ได้ ถ้าแบบจำลองถูกต้อง ส่วนเหลือควรเป็นสัญญาณรบกวนล้วน ๆ ถ้าแบบจำลองขาดอะไร สิ่งที่ขาดจะปรากฏเป็นรูปแบบที่มองเห็นได้ เหมือนแพทย์ที่ยังได้ยินเสียงผิดปกติเป็นจังหวะในปอดหลังให้ยา

| ภาพ | สิ่งที่ดู | สิ่งที่ตรวจ |
|---|---|---|
| ส่วนเหลือเทียบกับค่าทำนาย $\hat y$ | การกระจายมีขนาดคงที่หรือไม่ | ความแปรปรวนไม่คงที่ |
| ส่วนเหลือเทียบกับตัวแปรอิสระ $x$ | มีแนวโค้งหรือแนวโน้มหรือไม่ | พจน์ที่ขาดไปในแบบจำลอง |
| ส่วนเหลือเทียบกับลำดับเวลาของการวัด | จุดที่ติดกันมีค่าใกล้กันหรือไม่ | ความไม่อิสระ การเลื่อนของเครื่องมือ |
| ฮิสโทแกรมและภาพควอนไทล์–ควอนไทล์ | รูปร่างของการแจกแจง | สมมติฐานความเป็นเกาส์ |

เซลล์ถัดไปวาดภาพทั้งสี่แบบจากข้อมูลในบทนี้ ภาพที่ (ค) เป็นตัวอย่างเพิ่มเติม: ข้อมูลสปริงชุดเดิมที่วัดในลำดับสุ่ม และเซนเซอร์เลื่อนขึ้นช้า ๆ 0.08 N ตลอดการทดลอง

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(11, 7))

# (a) straight line on the anharmonic spring: residual vs x
lin_an = LinearRegression().fit(X_an, F_an)
ax[0, 0].plot(x_an, F_an - lin_an.predict(X_an), "o", ms=4); ax[0, 0].axhline(0, color="k", lw=0.8)
ax[0, 0].set_xlabel("x (m)"); ax[0, 0].set_ylabel("residual (N)")
ax[0, 0].set_title("(a) line on anharmonic spring: missing x^3 term")

# (b) diode: raw residual vs prediction (weighted fit)
I_hat = shockley(V, *popt_d)
ax[0, 1].semilogx(I_hat, I_obs - I_hat, "o", ms=4, label="raw residual (A)")
ax[0, 1].axhline(0, color="k", lw=0.8); ax[0, 1].set_xlabel("predicted I (A)"); ax[0, 1].set_ylabel("residual (A)")
ax[0, 1].set_title("(b) diode: spread grows with I (proportional noise)")
ins = ax[0, 1].inset_axes([0.08, 0.1, 0.4, 0.35])
ins.semilogx(I_hat, (I_obs - I_hat) / (0.05 * I_obs), "o", ms=2, color="C1"); ins.axhline(0, color="k", lw=0.6)
ins.set_title("residual / sigma_i", fontsize=7); ins.tick_params(labelsize=6)

# (c) Hooke data measured in random order with a slow sensor drift
rng_o = np.random.default_rng(7)
order = rng_o.permutation(x.size)                    # measurement order of the 50 stretches
F_drift = F_obs.copy(); F_drift[order] += 0.08 * np.arange(x.size) / (x.size - 1)
r_drift = F_drift - LinearRegression().fit(X, F_drift).predict(X)
ax[1, 0].plot(np.arange(x.size), r_drift[order], "o-", ms=3, lw=0.5)
ax[1, 0].axhline(0, color="k", lw=0.8); ax[1, 0].set_xlabel("measurement number"); ax[1, 0].set_ylabel("residual (N)")
ax[1, 0].set_title("(c) residual vs time: sensor drift")

# (d) damped oscillator: normal Q-Q plot of the residuals
(osm, osr), (slope_, icpt_, _) = stats.probplot(resid_d / sigma_x)
ax[1, 1].plot(osm, osr, "o", ms=3); ax[1, 1].plot(osm, slope_ * osm + icpt_, "r")
ax[1, 1].set_xlabel("theoretical quantile"); ax[1, 1].set_ylabel("residual / sigma")
ax[1, 1].set_title("(d) damped oscillator: Q-Q plot, Gaussian")
fig.suptitle("four residual plots (SYNTHETIC)"); plt.tight_layout(); plt.show()
print(f"(c) drift: corr(residual, measurement number) = {np.corrcoef(r_drift[order], np.arange(x.size))[0, 1]:+.2f}, "
      f"corr(residual, x) = {np.corrcoef(r_drift, x)[0, 1]:+.2f} (always 0 for OLS with an intercept)")

แต่ละรูปแบบบอกเรื่องทางฟิสิกส์ได้ (ก) แนวโค้งบอกว่าขาดพจน์ไม่เชิงเส้น (ข) ส่วนเหลือที่กระจายกว้างขึ้นตามขนาดของค่าที่วัดบอกว่าสัญญาณรบกวนเป็นแบบสัดส่วนและต้องถ่วงน้ำหนัก เมื่อหารด้วย $\sigma_i$ แล้ว (ภาพเล็ก) การกระจายจะคงที่ (ค) ส่วนเหลือที่ค่อย ๆ เลื่อนตามเวลาบอกว่าเครื่องมือเลื่อน ซึ่งมองไม่เห็นถ้าวาดเทียบกับ $x$ เท่านั้น เพราะสหสัมพันธ์ระหว่างส่วนเหลือของ OLS กับ $x$ เป็นศูนย์เสมอ (ง) จุดบนภาพควอนไทล์–ควอนไทล์อยู่บนเส้นตรง สอดคล้องกับสัญญาณรบกวนแบบเกาส์

บทเรียนคือ ส่วนเหลือที่ไม่สุ่มเป็นสัญญาณว่ามีฟิสิกส์ที่ขาดไป ไม่ใช่สัญญาณรบกวนที่ลดได้ด้วยการเก็บข้อมูลเพิ่ม การเก็บข้อมูลเพิ่มยิ่งทำให้รูปแบบชัดขึ้น เพราะสัญญาณรบกวนเฉลี่ยหักล้างกันไป แต่รูปแบบที่เป็นระบบไม่หาย แบบฝึกหัด E5.6 ในส่วนที่ 18 ใช้หลักนี้กับไดโอดที่มีความต้านทานอนุกรม

---
# ส่วนที่ 16 ไคกำลังสองลดทอน (หัวข้อ 5.5.2)

$$\chi^2_{\text{red}} = \frac{\chi^2_{\min}}{\nu},\qquad \nu = N - p$$

ถ้าแบบจำลองถูกต้องและ $\sigma_i$ เป็นความไม่แน่นอนจริง แต่ละพจน์ $(r_i/\sigma_i)^2$ มีค่าเฉลี่ยราว 1 ผลรวมจึงราว $\nu$ แต่ $\chi^2_{\min}$ เป็นตัวแปรสุ่มที่มีส่วนเบี่ยงเบนมาตรฐาน $\sqrt{2\nu}$ ดังนั้น $\chi^2_{\text{red}}$ กระจายรอบ 1 ด้วยส่วนเบี่ยงเบนมาตรฐานราว $\sqrt{2/\nu}$

| $\chi^2_{\text{red}}$ | ความหมายทางสถิติ | คำอธิบายทางฟิสิกส์ที่เป็นไปได้ |
|---|---|---|
| อยู่ใน $1 \pm 2\sqrt{2/\nu}$ | ส่วนเหลือสอดคล้องกับความไม่แน่นอนที่ระบุ | แบบจำลองและความไม่แน่นอนสอดคล้องกับข้อมูล |
| มากกว่า 1 อย่างมีนัยสำคัญ | ส่วนเหลือใหญ่กว่าความไม่แน่นอนที่ระบุ | แบบจำลองขาดกลไก หรือประเมิน $\sigma_i$ ต่ำเกินไป |
| น้อยกว่า 1 อย่างมีนัยสำคัญ | ส่วนเหลือเล็กกว่าความไม่แน่นอนที่ระบุ | ประเมิน $\sigma_i$ สูงเกินไป หรือพารามิเตอร์มากจนเรียนรู้เกิน |

In [ ]:
chi2_min = np.sum((resid_d / sigma_x)**2)
nu = t.size - len(popt)
print(f"damped oscillator: chi2 = {chi2_min:.1f}, nu = {nu}, chi2_red = {chi2_min / nu:.2f}, "
      f"sqrt(2/nu) = {np.sqrt(2 / nu):.2f}, p = {stats.chi2.sf(chi2_min, nu):.3f}")

ค่า 1.19 สูงกว่า 1 ราวสองเท่าของ $\sqrt{2/\nu}$ และค่าพีราว 0.035 ถ้าดูเพียงตัวเลขนี้ อาจสรุปว่าแบบจำลองมีปัญหา แต่ข้อมูลชุดนี้สร้างจากแบบจำลองที่ถูกต้องและ $\sigma$ ที่ถูกต้องทุกประการ เซลล์ถัดไปสร้างข้อมูลใหม่ 1,000 ชุดจากแบบจำลองเดียวกัน ปรับทุกชุด แล้วดูการแจกแจงของ $\chi^2_{\text{red}}$ (ใช้เวลาราว 5 วินาที)

In [ ]:
rng_c = np.random.default_rng(5)
b_osc = ([0, 0, 0, -np.pi], [10, 5, 20, np.pi])
chi2_sim = []
for _ in range(1000):
    xs = damped_oscillator(t, 1.0, 0.3, 6.0, 0.5) + rng_c.normal(scale=sigma_x, size=t.size)
    ps, _ = curve_fit(damped_oscillator, t, xs, p0=[1.0, 0.2, 6.0, 0.0], bounds=b_osc,
                      sigma=sigma_x * np.ones(t.size), absolute_sigma=True)
    chi2_sim.append(np.sum(((xs - damped_oscillator(t, *ps)) / sigma_x)**2) / nu)
chi2_sim = np.array(chi2_sim)
print(f"1,000 correct fits: chi2_red mean {chi2_sim.mean():.3f}, std {chi2_sim.std():.3f} (sqrt(2/nu) = {np.sqrt(2 / nu):.3f})")
print(f"fraction with chi2_red >= {chi2_min / nu:.2f}: {np.mean(chi2_sim >= chi2_min / nu):.3f}")

plt.figure(figsize=(6.5, 3.3))
plt.hist(chi2_sim, bins=40, density=True, alpha=0.6, label="1,000 simulated fits")
cg = np.linspace(0.6, 1.5, 300); plt.plot(cg, stats.chi2.pdf(cg * nu, nu) * nu, "k", label=r"$\chi^2_\nu/\nu$")
plt.axvline(chi2_min / nu, color="r", ls="--", label="our data set")
plt.xlabel(r"$\chi^2_{\rm red}$"); plt.legend(fontsize=8); plt.title("chi2_red of a correct model (SYNTHETIC)")
plt.tight_layout(); plt.show()

ราวสามถึงสี่ครั้งในร้อยครั้ง แบบจำลองที่ถูกต้องก็ให้ $\chi^2_{\text{red}}$ สูงเท่านี้ ค่า $\chi^2_{\text{red}}$ ที่ห่างจาก 1 เล็กน้อยจึงต้องอ่านคู่กับภาพของส่วนเหลือเสมอ ถ้าส่วนเหลือไม่มีรูปแบบ (ส่วนที่ 12 และภาพ (ง) ของส่วนที่ 15) ค่า 1.19 ที่ $\nu = 196$ ยังไม่ใช่หลักฐานที่หนักแน่นพอจะเปลี่ยนแบบจำลอง

> **ความเข้าใจคลาดเคลื่อนที่พบบ่อย:** "$\chi^2_{\text{red}}$ ยิ่งน้อยยิ่งดี" ค่าที่ดีคือค่าที่ใกล้ 1 ถ้าได้ $\chi^2_{\text{red}} = 0.1$ จากข้อมูลร้อยจุด ส่วนเหลือเล็กกว่าความไม่แน่นอนที่อ้างราว $\sqrt{10} \approx 3.2$ เท่า ซึ่งแทบเป็นไปไม่ได้โดยบังเอิญ (เซลล์ถัดไป) คำอธิบายที่เป็นไปได้มากที่สุดคือประเมินความไม่แน่นอนสูงเกินไป

In [ ]:
nu100 = 100 - 2
print(f"chi2_red = 0.1 with 100 points (nu = {nu100}): P(chi2_red <= 0.1) = {stats.chi2.cdf(0.1 * nu100, nu100):.0e}")

### ✏️ แบบฝึก 5.F ค่าเดียวกัน ความหมายต่างกัน (คำถามทบทวนที่ 6)
การปรับหนึ่งได้ $\chi^2_{\text{red}} = 1.3$ ด้วย $\nu = 10$ อีกการปรับหนึ่งได้ $\chi^2_{\text{red}} = 1.3$ ด้วย $\nu = 1000$ คำนวณว่าแต่ละค่าห่างจาก 1 กี่เท่าของ $\sqrt{2/\nu}$ เก็บใน `my_dev10` และ `my_dev1000` แล้วระบุใน `my_suspicious` ว่ากรณีใดเป็นหลักฐานว่าแบบจำลองขาดบางอย่าง (`10` หรือ `1000`)

In [ ]:
my_dev10 = None         # ✏️ (1.3 - 1) / sqrt(2/10)
my_dev1000 = None       # ✏️
my_suspicious = None    # ✏️ 10 หรือ 1000

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_dev10', 'my_dev1000', 'my_suspicious'])
_d = {10: 0.3 / np.sqrt(2 / 10), 1000: 0.3 / np.sqrt(2 / 1000)}
for nu_, mine in ((10, my_dev10), (1000, my_dev1000)):
    _say(f"nu = {nu_:4d}:", "(ยังไม่ได้ตอบ)" if mine is None else ("✔" if abs(mine - _d[nu_]) < 0.05 else "✘"),
          f"— ห่าง {_d[nu_]:.1f} sigma, P(chi2_red >= 1.3) = {stats.chi2.sf(1.3 * nu_, nu_):.1e}")
_say("กรณีที่น่าสงสัย:", "(ยังไม่ได้ตอบ)" if my_suspicious is None else ("✔" if my_suspicious == 1000 else "✘"),
      "— nu = 1000 ห่างจาก 1 ราว 6.7 เท่าของ sqrt(2/nu) ส่วน nu = 10 ห่างไม่ถึงหนึ่งเท่า")

---
# ส่วนที่ 17 การแพร่ของความคลาดเคลื่อนด้วยเมทริกซ์ความแปรปรวนร่วม (หัวข้อ 5.5.3)

พารามิเตอร์ที่ได้จากการปรับเส้นโค้งมักสัมพันธ์กัน เช่น $I_0$ กับ $n$ ของไดโอด ($\rho \approx 0.98$) หรือจุดตัดแกนกับความชันของข้อมูลสปริง ($\rho \approx -0.86$ เหมือนไม้กระดก) สำหรับปริมาณ $f(\boldsymbol\theta)$ ที่คำนวณจากพารามิเตอร์ สูตรที่ถูกต้องคือ

$$\sigma_f^2 = \mathbf{J}^\top\boldsymbol\Sigma\,\mathbf{J} = \sum_{i,j}\frac{\partial f}{\partial\theta_i}\frac{\partial f}{\partial\theta_j}\Sigma_{ij}$$

เมื่อ $\boldsymbol\Sigma$ คือ `pcov` และ $\mathbf{J}$ คือเวกเตอร์ของอนุพันธ์ย่อย พจน์ $i = j$ คือสูตรที่ไม่คิดสหสัมพันธ์ ส่วนพจน์ $i \ne j$ คือพจน์ความแปรปรวนร่วม

| กรณี | ผลต่อ $\sigma_f$ เทียบกับสูตรที่ไม่คิดสหสัมพันธ์ |
|---|---|
| $\rho_{ij} \approx 0$ | แทบไม่ต่าง |
| $\rho_{ij}\,\dfrac{\partial f}{\partial\theta_i}\dfrac{\partial f}{\partial\theta_j} > 0$ | $\sigma_f$ จริง **มากกว่า** สูตรแบบง่ายประเมินต่ำเกินไป |
| $\rho_{ij}\,\dfrac{\partial f}{\partial\theta_i}\dfrac{\partial f}{\partial\theta_j} < 0$ | $\sigma_f$ จริง **น้อยกว่า** สูตรแบบง่ายประเมินสูงเกินไป |

สำหรับ $Q = \omega/(2\gamma)$ อนุพันธ์คือ $\partial Q/\partial\omega = 1/(2\gamma)$ และ $\partial Q/\partial\gamma = -\omega/(2\gamma^2)$ โค้ดที่ 5.8 ใช้ผลจากโค้ดที่ 5.6 ซึ่งเรียงพารามิเตอร์เป็น $(A, \gamma, \omega, \phi)$

### โค้ดที่ 5.8 — การแพร่ความไม่แน่นอนแบบไม่คิดและแบบคิดสหสัมพันธ์

In [ ]:
Q = omega / (2 * gamma)
dQ_dom = 1 / (2 * gamma)
dQ_dga = -omega / (2 * gamma**2)

# naive: ignore the off-diagonal term
s_naive = np.sqrt(dQ_dom**2 * pcov[2, 2] + dQ_dga**2 * pcov[1, 1])

# full: include the covariance between gamma (index 1) and omega (index 2)
s_full = np.sqrt(dQ_dom**2 * pcov[2, 2] + dQ_dga**2 * pcov[1, 1]
                 + 2 * dQ_dom * dQ_dga * pcov[1, 2])

# matrix form, J ordered as (A, gamma, omega, phi)
J = np.array([0.0, dQ_dga, dQ_dom, 0.0])
s_matrix = np.sqrt(J @ pcov @ J)       # identical to s_full

rho = pcov / np.outer(errs, errs)      # correlation matrix

In [ ]:
print(f"Q = {Q:.2f}; sigma_Q naive = {s_naive:.3f}, full = {s_full:.3f}, matrix = {s_matrix:.3f}  ->  Q = 10.5 +/- 0.3")
print("correlation matrix (A, gamma, omega, phi):")
print(pd.DataFrame(rho, index=["A", "gamma", "omega", "phi"], columns=["A", "gamma", "omega", "phi"]).round(2))

ความไม่แน่นอนทั้งสองแบบแทบเท่ากัน เพราะ $\gamma$ กับ $\omega$ แทบไม่สัมพันธ์กัน ($\rho \approx -0.05$) การแกว่งกำหนดความถี่ และการลดลงของแอมพลิจูดกำหนดอัตราการหน่วง ข้อมูลสองส่วนนี้แยกจากกันได้ดี

แต่ $A$ กับ $\gamma$ สัมพันธ์กันสูง ($\rho \approx +0.73$) ถ้าเริ่มต้นสูงขึ้น ก็ต้องลดลงเร็วขึ้นจึงจะเข้ากับข้อมูลช่วงหลังได้ ลองพิจารณาแอมพลิจูดที่เวลา 5 s คือ $a = Ae^{-5\gamma}$ อนุพันธ์เทียบกับ $A$ เป็นบวก เทียบกับ $\gamma$ เป็นลบ และ $\rho$ เป็นบวก ผลคูณจึงเป็นลบ เซลล์ถัดไปคำนวณทั้งสองแบบ แล้วตรวจกับการสุ่มจากการแจกแจงเกาส์หลายตัวแปร $\mathcal{N}(\hat{\boldsymbol\theta}, \boldsymbol\Sigma)$

In [ ]:
t5 = 5.0
a5 = A * np.exp(-gamma * t5)
Ja = np.array([np.exp(-gamma * t5), -t5 * A * np.exp(-gamma * t5), 0.0, 0.0])
s_a_naive = np.sqrt(np.sum(Ja**2 * np.diag(pcov)))
s_a_full = np.sqrt(Ja @ pcov @ Ja)
print(f"a(5 s) = {a5:.3f} m; sigma naive = {s_a_naive:.4f} m, full = {s_a_full:.4f} m (naive / full = {s_a_naive / s_a_full:.2f})")

draws = np.random.default_rng(1).multivariate_normal(popt, pcov, size=200_000)
print(f"Monte Carlo from N(popt, pcov): sigma_Q = {np.std(draws[:, 2] / (2 * draws[:, 1])):.3f}, "
      f"sigma_a(5 s) = {np.std(draws[:, 0] * np.exp(-5 * draws[:, 1])):.4f} m")

plt.figure(figsize=(5, 4))
plt.plot(draws[:3000, 0], draws[:3000, 1], ".", ms=2, alpha=0.4)
Ag = np.linspace(draws[:, 0].min(), draws[:, 0].max(), 50)
for lvl in (a5 - s_a_full, a5, a5 + s_a_full):
    plt.plot(Ag, np.log(Ag / lvl) / 5, "r", lw=1)
plt.xlabel("A (m)"); plt.ylabel(r"$\gamma$ (1/s)"); plt.title("A-gamma draws and lines of constant a(5 s)")
plt.tight_layout(); plt.show()

สูตรแบบง่ายประเมินความไม่แน่นอนของ $a(5\,\text{s})$ สูงเกินไปราวครึ่งหนึ่ง ภาพแสดงเหตุผล จุดที่สุ่มจาก $(A, \gamma)$ เรียงตัวตามแนวเดียวกับเส้นที่ $a(5\,\text{s})$ คงที่ (เส้นสีแดง) ข้อมูลช่วงกลางกำหนดแอมพลิจูดที่ 5 s ได้แม่นยำ แม้จะแยกไม่ค่อยได้ว่าส่วนใดมาจาก $A$ และส่วนใดมาจาก $\gamma$ การสุ่มแบบมอนติคาร์โลให้ผลตรงกับสูตรเต็ม

หลักปฏิบัติคือ อย่าตัดสินล่วงหน้าว่าสหสัมพันธ์มีผลหรือไม่ ให้ใช้รูปเมทริกซ์ $\mathbf{J}^\top\boldsymbol\Sigma\mathbf{J}$ เสมอ สูตรนี้อาศัยการประมาณเชิงเส้นของ $f$ รอบค่าที่ดีที่สุด ถ้า $f$ ไม่เป็นเชิงเส้นมากในช่วงความไม่แน่นอน วิธีบูตสแตรปและวิธีแบบเบส์ในบทที่ 9 จะเชื่อถือได้กว่า

### ✏️ แบบฝึก 5.G ความไม่แน่นอนของค่าทำนายจากเส้นตรง (คำถามทบทวนที่ 7)
ความชัน $m$ และจุดตัดแกน $c$ ของข้อมูลสปริง (โค้ดที่ 5.2) มีสหสัมพันธ์เป็นลบ สำหรับค่าทำนาย $\hat y = c + mx_0$ ที่ $x_0 > 0$ ให้ตอบว่าการละเลยสหสัมพันธ์จะประเมินความไม่แน่นอน `"สูงเกินไป"` หรือ `"ต่ำเกินไป"` แล้วคำนวณ $\sigma_{\hat y}$ ทั้งสองแบบที่ $x_0 = 0.10$ m ด้วย `C = ols_model.cov_params()` และ $\mathbf{J} = (1, x_0)$

In [ ]:
C = ols_model.cov_params()
x0 = 0.10
my_direction = "?"       # ✏️ "สูงเกินไป" หรือ "ต่ำเกินไป"
my_naive = None          # ✏️ sqrt(C[0,0] + x0**2 C[1,1])
my_full = None           # ✏️ sqrt(J @ C @ J)

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_direction', 'my_full', 'my_naive'])
_say("ทิศทาง:", "(ยังไม่ได้ตอบ)" if my_direction == "?" else ("✔" if my_direction == "สูงเกินไป" else "✘"),
      "— อนุพันธ์ 1 และ x0 เป็นบวกทั้งคู่ คูณกับ rho < 0 พจน์ความแปรปรวนร่วมจึงเป็นลบ")
_say(f"corr(c, m) = {C[0, 1] / np.sqrt(C[0, 0] * C[1, 1]):.2f}")
for x0_ in (0.0, 0.05, 0.10, 0.15, 0.20):
    J_ = np.array([1.0, x0_])
    nv, fl = np.sqrt(np.sum(J_**2 * np.diag(C))), np.sqrt(J_ @ C @ J_)
    mark = ""
    if x0_ == 0.10 and my_naive is not None and my_full is not None:
        mark = "  ✔" if abs(my_naive - nv) < 1e-4 and abs(my_full - fl) < 1e-4 else "  ✘ ตรวจสูตรอีกครั้ง"
    _say(f"   x0 = {x0_:.2f} m: naive {nv:.4f} N, full {fl:.4f} N, naive/full = {nv / fl:.2f}{mark}")
_say(f"ผลต่างมากที่สุดใกล้ค่าเฉลี่ยของ x ({x.mean():.2f} m) ซึ่งค่าทำนายมีความไม่แน่นอนต่ำที่สุด")

### เชื่อมไปยังบทที่ 9
ฟังก์ชัน $\chi^2$ จะกลายเป็นฟังก์ชันภาวะน่าจะเป็นในกรอบแบบเบส์ ขอบเขตทางฟิสิกส์จะกลายเป็นการแจกแจงก่อนหน้า (prior distribution) บทที่ 9 จะนำออสซิลเลเตอร์ของ E5.5 มาวิเคราะห์ซ้ำด้วยวิธีแบบเบส์ การแพร่ความไม่แน่นอนด้วยอนุพันธ์จะเทียบกับบูตสแตรป และการตรวจสอบไขว้จะขยายเป็นกรอบการเลือกแบบจำลองที่สมบูรณ์

---
# ส่วนที่ 18 งานนอกชั้น: E5.2 และ E5.5 ส่วนที่เหลือ, E5.4, E5.6 และ E5.7

ส่วนนี้ทำนอกเวลาเรียน (รวมราว 1.5–2 ชั่วโมงตามคู่มือผู้สอน) ต้องรันส่วนที่ 0–17 ก่อน ถ้าเปิดโน้ตบุ๊กใหม่ ให้คลิกเซลล์นี้แล้วเลือก Runtime → Run before

### E5.5 ส่วนที่เหลือ: รายงานผลการปรับเส้นโค้งของออสซิลเลเตอร์แบบหน่วง (20 นาที)
ในชั้นเรียนได้ประมาณค่าเริ่มต้นจากกราฟและปรับเส้นโค้งแล้ว (ส่วนที่ 13) งานนอกชั้นคือเขียนรายงานสั้นที่มี (1) วิธีประมาณค่าเริ่มต้นของ $A$, $\gamma$ และ $\omega$ จากกราฟ (2) ค่า $\gamma$ และ $\omega$ พร้อมความคลาดเคลื่อนมาตรฐานและหน่วย (3) ค่า $Q$ เทียบกับค่าจริงที่ใช้สร้างข้อมูล และ (4) กราฟส่วนเหลือเทียบกับเวลา พร้อมคำอธิบายว่ามีรูปแบบหรือไม่ และทำไมความคลาดเคลื่อนสัมพัทธ์ของ $\omega$ จึงเล็กกว่าของ $\gamma$ มาก

### ✏️ E5.2 ส่วนที่สอง ทำไมการปรับลงโทษต้องปรับมาตราส่วนก่อน (20 นาที)
ใช้ข้อมูลสปริงจากโค้ดที่ 5.1 ปรับการถดถอยริดจ์ `Ridge(alpha=1.0)` สองครั้ง ครั้งแรกด้วย $x$ ในหน่วยเมตร ครั้งที่สองด้วย $x$ ในหน่วยมิลลิเมตร (`X * 1000`) แล้วแปลงความชันของทั้งสองครั้งกลับเป็น N/m เก็บใน `my_ridge_m` และ `my_ridge_mm` จากนั้นอธิบายผลในเซลล์ข้อความ

In [ ]:
my_ridge_m = None       # ✏️ Ridge(alpha=1.0).fit(X, F_obs).coef_[0]
my_ridge_mm = None      # ✏️ Ridge(alpha=1.0).fit(X * 1000, F_obs).coef_[0] * 1000

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_ridge_m', 'my_ridge_mm'])
_rm = Ridge(alpha=1.0).fit(X, F_obs).coef_[0]
_rmm = Ridge(alpha=1.0).fit(X * 1000, F_obs).coef_[0] * 1000
for name, mine, ref in (("x in m ", my_ridge_m, _rm), ("x in mm", my_ridge_mm, _rmm)):
    _say(name, "(ยังไม่ได้ตอบ)" if mine is None else ("✔" if abs(mine / ref - 1) < 1e-4 else "✘"),
          f"— slope converted back to N/m = {ref:.3f}")
_say(f"OLS slope = {model.coef_[0]:.3f} N/m -> the same penalty shrinks k strongly in metres and almost not at all in millimetres")

**คำอธิบาย (E5.2):** ความชันจากข้อมูลดิบกับจากข้อมูลที่ปรับเป็นมาตรฐานสัมพันธ์กันอย่างไร และทำไมการปรับลงโทษจึงขึ้นกับหน่วย

...

### ✏️ E5.4 การเลือก λ ด้วยการตรวจสอบไขว้ (20 นาที)
ทำโค้ดที่ 5.5 ซ้ำกับข้อมูลที่สัญญาณรบกวนเพิ่มเป็น 0.15 N (ค่า $x$ ชุดเดิม) แล้ววาดคะแนนการตรวจสอบไขว้ของทั้งสองระดับสัญญาณรบกวนบนแกนลอการิทึมกราฟเดียวกัน เก็บ $\lambda^*$ และคะแนนต่ำสุดของกรณี 0.15 N ใน `my_lambda_015` และ `my_cvmin_015` แล้วอธิบายทิศทางที่ $\lambda^*$ เปลี่ยน โดยเชื่อมกับ $\lambda = \sigma^2/\tau^2$

In [ ]:
def cv_curve(noise):
    r = np.random.default_rng(seed=2026)                 # same draws as code 5.4
    xs = r.uniform(-0.15, 0.15, 60)
    Fs = 5.0 * xs + 500.0 * xs**3 + r.normal(scale=noise, size=xs.size)
    out = []
    for lam in lambdas:
        # ✏️ fill in: 5-fold CV MSE of poly10(Ridge(alpha=lam)) on (xs.reshape(-1, 1), Fs) with cv=kfold
        out.append(np.nan)
    return np.array(out)

cv05, cv15 = cv_curve(0.05), cv_curve(0.15)
my_lambda_015 = None     # ✏️ lambdas[np.nanargmin(cv15)]
my_cvmin_015 = None      # ✏️

In [ ]:
#@markdown ✅ ตรวจคำตอบ (and the reference plot)
_say = _gate(['my_cvmin_015', 'my_lambda_015'])
def _cv_ref(noise):
    r = np.random.default_rng(seed=2026)
    xs = r.uniform(-0.15, 0.15, 60)
    Fs = 5.0 * xs + 500.0 * xs**3 + r.normal(scale=noise, size=xs.size)
    return np.array([-cross_val_score(poly10(Ridge(alpha=lam)), xs.reshape(-1, 1), Fs, cv=kfold,
                                      scoring="neg_mean_squared_error").mean() for lam in lambdas])
_c5, _c15 = _cv_ref(0.05), _cv_ref(0.15)
for noise, c in ((0.05, _c5), (0.15, _c15)):
    k_ = np.argmin(c); fl = lambdas[c < 1.1 * c[k_]]
    _say(f"noise {noise:.2f} N: lambda* = {lambdas[k_]:.3f}, min CV MSE = {c[k_]:.4f} N^2 "
          f"(sigma^2 = {noise**2:.4f}); within 10 % of the minimum: {fl.min():.3f}-{fl.max():.3f}")
_say("lambda* (0.15 N):", "(ยังไม่ได้ตอบ)" if my_lambda_015 is None else
      ("✔" if abs(my_lambda_015 / lambdas[np.argmin(_c15)] - 1) < 1e-6 else "✘"))
_say("min CV MSE (0.15 N):", "(ยังไม่ได้ตอบ)" if my_cvmin_015 is None else
      ("✔" if abs(my_cvmin_015 / _c15.min() - 1) < 0.01 else "✘"))
plt.figure(figsize=(6, 3.6))
plt.loglog(lambdas, _c5, "o-", label="noise 0.05 N"); plt.loglog(lambdas, _c15, "s-", label="noise 0.15 N")
plt.xlabel(r"$\lambda$"); plt.ylabel("5-fold CV MSE"); plt.legend(); plt.title("E5.4 reference (SYNTHETIC)")
plt.tight_layout(); plt.show()

**คำอธิบาย (E5.4):** $\lambda^*$ เลื่อนไปทางใด ทำไมไม่เลื่อนเก้าเท่าตามอัตราส่วนของความแปรปรวน และทำไมการรายงาน $\lambda^*$ ละเอียดหลายหลักจึงไม่มีความหมาย

...

### ✏️ E5.6 การวิเคราะห์ส่วนเหลือของไดโอด (25 นาที)
ข้อมูลกระแสและแรงดันของไดโอดชุดใหม่วัดที่ 46 แรงดันในช่วง 0.30–0.75 V ด้วยสัญญาณรบกวนร้อยละ 5 ของกระแส เซลล์ซ่อนถัดไปสร้างข้อมูลไว้ในไฟล์ `data/diode_series_R.csv` (อย่าเพิ่งเปิดดูโค้ดจนกว่าจะทำเสร็จ เพราะโค้ดบอกคำตอบ)

1. ปรับสมการของช็อกลีย์ในอุดมคติด้วยวิธีเดียวกับโค้ดที่ 5.7 (`sigma = 0.05 * I`)
2. คำนวณส่วนเหลือที่หารด้วยความไม่แน่นอน $(I_{\text{obs}} - \hat I)/\sigma_i$ แล้ววาดเทียบกับ $V$
3. คำนวณ $\chi^2_{\text{red}}$ เก็บใน `my_chi2red_ideal` และค่า $n$ ใน `my_n_ideal` แล้วตัดสินว่าแบบจำลองเพียงพอหรือไม่
4. ถ้าไม่เพียงพอ เสนอกลไกทางฟิสิกส์ที่ขาดไปและอธิบายรูปแบบของส่วนเหลือ แล้วปรับแบบจำลองที่แก้แล้วด้วยฟังก์ชัน `shockley_rs(V, I0, n, Rs)` ที่เซลล์ซ่อนเตรียมไว้ (แก้สมการโดยนัย $I = I_0\{\exp[(V - IR_s)/(nV_T)] - 1\}$ ด้วยฟังก์ชันแลมเบิร์ต $W$) เก็บ $R_s$ ใน `my_Rs` (โอห์ม)

In [ ]:
#@title สร้างข้อมูลไดโอดของ E5.6 และฟังก์ชัน shockley_rs (ดับเบิลคลิกเพื่อดูโค้ด)
# โค้ดชุดเดียวกับ ch05_regression/exercises/E5_6_diode_series_resistance.py
from scipy.special import lambertw

def shockley_rs(V, I0, n, Rs):
    """Exact solution of I = I0 (exp((V - I Rs)/(n V_T)) - 1) via Lambert W."""
    a = n * V_T
    arg = I0 * Rs / a * np.exp((V + I0 * Rs) / a)
    return a / Rs * np.real(lambertw(arg)) - I0


rng_e6 = np.random.default_rng(seed=2026)
V6 = np.linspace(0.30, 0.75, 46)
I6_true = shockley_rs(V6, 2e-12, 1.4, 20.0)
I6_obs = I6_true * (1 + rng_e6.normal(scale=0.05, size=V6.size))
if not os.path.exists('data/diode_series_R.csv'):
    pd.DataFrame({'V': V6, 'I_A': I6_obs}).to_csv('data/diode_series_R.csv', index=False)
del V6, I6_true, I6_obs, rng_e6
print('data/diode_series_R.csv พร้อมใช้ (แถว, หลัก):', pd.read_csv('data/diode_series_R.csv').shape)

In [ ]:
d6 = pd.read_csv("data/diode_series_R.csv")
V6, I6 = d6["V"].to_numpy(), d6["I_A"].to_numpy()
sig6 = 0.05 * I6

# 1-3. ideal Shockley fit
p6 = None               # ✏️ curve_fit(shockley, V6, I6, p0=[1e-12, 1.5], bounds=([1e-15, 1.0], [1e-9, 2.0]), sigma=sig6, absolute_sigma=True)[0]
my_n_ideal = None       # ✏️
my_chi2red_ideal = None # ✏️ sum(z**2) / (46 - 2), z = (I6 - shockley(V6, *p6)) / sig6

# 4. model with a series resistance
p6rs = None             # ✏️ curve_fit(shockley_rs, V6, I6, p0=[1e-12, 1.5, 10.0], bounds=([1e-15, 1.0, 0.0], [1e-9, 2.0, 1e3]), sigma=sig6, absolute_sigma=True)[0]
my_Rs = None            # ✏️

In [ ]:
#@markdown ✅ ตรวจคำตอบ (and the reference residual plot)
_say = _gate(['my_Rs', 'my_chi2red_ideal', 'my_n_ideal'])
_b = ([1e-15, 1.0], [1e-9, 2.0])
_p, _ = curve_fit(shockley, V6, I6, p0=[1e-12, 1.5], bounds=_b, sigma=sig6, absolute_sigma=True)
_z = (I6 - shockley(V6, *_p)) / sig6
_p3, _c3 = curve_fit(shockley_rs, V6, I6, p0=[1e-12, 1.5, 10.0], bounds=([1e-15, 1.0, 0.0], [1e-9, 2.0, 1e3]),
                     sigma=sig6, absolute_sigma=True)
_z3 = (I6 - shockley_rs(V6, *_p3)) / sig6
_nu6 = V6.size - 2
_say("n (ideal):", "(ยังไม่ได้ตอบ)" if my_n_ideal is None else ("✔" if abs(my_n_ideal - _p[1]) < 2e-3 else "✘"),
      f"— n = {_p[1]:.3f} (true 1.4): biased because the model lacks R_s")
_say("chi2_red (ideal):", "(ยังไม่ได้ตอบ)" if my_chi2red_ideal is None else
      ("✔" if abs(my_chi2red_ideal - np.sum(_z**2) / _nu6) < 0.05 else "✘"),
      f"— {np.sum(_z**2) / _nu6:.1f} at nu = {_nu6}; sqrt(2/nu) = {np.sqrt(2 / _nu6):.2f} -> the model is not adequate")
_say("R_s:", "(ยังไม่ได้ตอบ)" if my_Rs is None else ("✔" if abs(my_Rs - _p3[2]) < 0.5 else "✘"),
      f"— R_s = {_p3[2]:.1f} +/- {np.sqrt(_c3[2, 2]):.1f} ohm, n = {_p3[1]:.3f}, chi2_red = {np.sum(_z3**2) / (V6.size - 3):.2f}")
plt.figure(figsize=(6.5, 3.4))
plt.plot(V6, _z, "o", ms=4, label="ideal Shockley"); plt.plot(V6, _z3, "s", ms=3, label="with $R_s$")
plt.axhline(0, color="k", lw=0.8); plt.xlabel("V (V)"); plt.ylabel(r"$(I_{obs}-\hat I)/\sigma$"); plt.legend()
plt.title("E5.6 reference residuals (SYNTHETIC)"); plt.tight_layout(); plt.show()

**คำอธิบาย (E5.6):** กลไกที่ขาดไปคืออะไร ทำให้ส่วนเหลือมีรูปแบบอย่างไร ทำไม $n$ จึงเอนเอียง และถ้าวัดเฉพาะแรงดันต่ำ (≤ 0.55 V) จะเห็นกลไกนี้หรือไม่

...

### ✏️ E5.7 การแพร่ความไม่แน่นอนไปยังตัวประกอบคุณภาพ (30 นาที)
1. ใช้ผลจาก E5.5 (ตัวแปร `popt`, `pcov` ของโค้ดที่ 5.6) คำนวณ $Q$ และ $\sigma_Q$ ทั้งแบบไม่คิดและแบบคิดสหสัมพันธ์ด้วยโค้ดที่ 5.8 แล้วรายงาน $Q$ ในรูปแบบที่ถูกต้อง เก็บเป็นข้อความใน `my_Q_report` เช่น `"Q = ... ± ..."`
2. เลือกปริมาณที่คำนวณต่อได้อีกหนึ่งปริมาณที่ความต่างระหว่างสองวิธีมีผลชัดเจน ตัวอย่างที่แนะนำคือเวลาของยอดคลื่นลูกแรกของตัวประกอบโคไซน์ $t_1 = (2\pi - \phi)/\omega$ ซึ่งขึ้นกับคู่ $\omega$ กับ $\phi$ ($\rho \approx -0.71$) เขียนเวกเตอร์ $\mathbf{J}$ ในลำดับ $(A, \gamma, \omega, \phi)$ แล้วเก็บ $\sigma_{t_1}$ ทั้งสองแบบใน `my_t1_naive` และ `my_t1_full`

In [ ]:
my_Q_report = "?"       # ✏️
t1 = (2 * np.pi - phi) / omega
J_t1 = None             # ✏️ np.array([dt/dA, dt/dgamma, dt/domega, dt/dphi])
my_t1_naive = None      # ✏️
my_t1_full = None       # ✏️

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_Q_report', 'my_t1_full', 'my_t1_naive'])
def _compare(Jv):
    return np.sqrt(np.sum(Jv**2 * np.diag(pcov))), np.sqrt(Jv @ pcov @ Jv)
rep = my_Q_report.replace(" ", "").replace("+/-", "±")
_say("รายงาน Q:", "(ยังไม่ได้ตอบ)" if my_Q_report == "?" else ("✔" if "10.5±0.3" in rep else "✘ รูปแบบที่ถูกคือ Q = 10.5 ± 0.3"))
_nt, _ft = _compare(np.array([0.0, 0.0, -t1 / omega, -1 / omega]))
if my_t1_naive is None or my_t1_full is None:
    _say("sigma_t1: (ยังไม่ได้ตอบ)")
else:
    _say("sigma_t1:", "✔" if abs(my_t1_naive - _nt) < 1e-5 and abs(my_t1_full - _ft) < 1e-5 else "✘ ตรวจอนุพันธ์อีกครั้ง")
_say(f"\n{'quantity':28s}  value     naive    full   naive/full")
rows_ = [("Q = omega/(2 gamma)", omega / (2 * gamma), np.array([0, -omega / (2 * gamma**2), 1 / (2 * gamma), 0])),
         ("a(5 s) = A exp(-5 gamma) [m]", A * np.exp(-5 * gamma), np.array([np.exp(-5 * gamma), -5 * A * np.exp(-5 * gamma), 0, 0]))]
for k_ in (1, 9):
    tk = (2 * np.pi * k_ - phi) / omega
    rows_.append((f"crest time t_{k_} [s]", tk, np.array([0, 0, -tk / omega, -1 / omega])))
for name, val, Jv in rows_:
    nv, fl = _compare(Jv)
    _say(f"{name:28s} {val:8.4f}  {nv:.4f}  {fl:.4f}   {nv / fl:.2f}")

**คำอธิบาย (E5.7):** ทำไม $\sigma_Q$ สองแบบจึงแทบเท่ากัน แต่ $\sigma$ ของ $a(5\,\text{s})$ และของเวลายอดคลื่นต่างกันชัด ใช้เครื่องหมายของ $\rho_{ij}\,\partial f/\partial\theta_i\,\partial f/\partial\theta_j$ ในการอธิบาย

...

---
### สรุปบท
วิธีกำลังสองน้อยสุดสามัญเป็นการประมาณภาวะน่าจะเป็นสูงสุดเมื่อสัญญาณรบกวนเป็นแบบเกาส์ และเป็นตัวประมาณเชิงเส้นที่ดีที่สุดภายใต้เงื่อนไขของเกาส์–มาร์คอฟแม้สัญญาณรบกวนไม่เป็นแบบเกาส์ ค่าสัมประสิทธิ์ต้องรายงานพร้อมความคลาดเคลื่อนมาตรฐานและตีความในเชิงฟิสิกส์ ส่วน $R^2$ ที่สูงไม่ได้รับประกันทั้งความแม่นยำและความถูกต้องของแบบจำลอง ลำดับขั้นหกขั้นและไปป์ไลน์จัดระเบียบการทำงานและป้องกันการรั่วไหลของข้อมูล ความซับซ้อนที่เหมาะสมเลือกได้ด้วยการตรวจสอบไขว้ ทั้งโดยเลือกดีกรีและโดยการปรับลงโทษ สำหรับแบบจำลองไม่เชิงเส้น การปรับด้วยไคกำลังสองใส่ขอบเขตทางฟิสิกส์ได้ ส่วนเหลือบอกว่าแบบจำลองขาดอะไร $\chi^2_{\text{red}}$ บอกว่าส่วนเหลือสอดคล้องกับความไม่แน่นอนหรือไม่ และเมทริกซ์ความแปรปรวนร่วมช่วยแพร่ความไม่แน่นอนได้ถูกต้อง ซึ่งสหสัมพันธ์อาจทำให้ความไม่แน่นอนเพิ่มหรือลดก็ได้

### เอกสารอ้างอิง
- Pedregosa, F., Varoquaux, G., Gramfort, A., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html
- Virtanen, P., Gommers, R., Oliphant, T. E., et al. (2020). SciPy 1.0: Fundamental algorithms for scientific computing in Python. *Nature Methods, 17*(3), 261–272. https://doi.org/10.1038/s41592-019-0686-2
- Carleo, G., Cirac, I., Cranmer, K., et al. (2019). Machine learning and the physical sciences. *Reviews of Modern Physics, 91*(4), 045002. https://doi.org/10.1103/RevModPhys.91.045002